# SparseNet-MDA-Cancer: reproducible analysis

**Question.** Does a *co-expression-module prior* improve miRNA–disease association prediction **more in an
association-sparse cancer (cholangiocarcinoma)** than in an **association-rich cancer (breast neoplasms)**?

**Idea in one paragraph.** For each cancer we (1) build a gene co-expression network from tumor-vs-control expression data,
(2) find gene modules whose activity tracks tumor status, (3) Find which miRNAs have experimentally validated targets
concentrated in those modules (hypergeometric enrichment), and (4) turn that into a *prior score per miRNA*.
We then add the prior as an extra feature to a baseline classifier (random forest or logistic regression trained on each miRNA's associations with other diseases) and measure **ΔAUPRC = AUPRC(prior) − AUPRC(baseline)** on held-out known miRNA–disease pairs.
The prior is built **only from expression + target data, never from the held-out association labels**, so it cannot leak.

## Workflow map (each notebook cell belongs to one step)

| Step | What happens | Cell(s) | Manuscript section |
|---|---|---|---|
| 0 | Settings: seeds, thresholds, dataset accessions | 2 | Methods (all) |
| 1 | Get the data: GEO expression and platform annotation, OmniPath targets, your HMDD file | 3 (functions), 11–12 | Materials and Data Sources |
| 2 | Audit HMDD pair counts, standardize miRNA names, build the association matrix | 4 (functions), 10 | Authenticated Data Audit |
| 3 | Label tumor/control, prepare expression, sample QC | 12, 13 | Dataset-Preparation Procedure |
| 4 | Build the co-expression network and gene modules | 5 (functions), 14 | Proposed Method: soft adjacency, TOM, modules |
| 5 | Link modules to tumor status (eigengene–trait correlation) | 14 | Module Eigengene–Trait Association |
| 6 | miRNA–module enrichment and the continuous prior | 6 (functions), 15 | Candidate Master-Regulator Enrichment |
| 7 | Positive–unlabelled evaluation: AUPRC(baseline) and AUPRC(prior), 30 seeds | 7 (functions), 16 | Experimental Design, Leakage Prevention |
| 8 | **ΔAUPRC** = AUPRC(prior) − AUPRC(baseline), per cancer, with bootstrap CI and paired test | 8 (functions), 17 | Evaluation Metrics and Statistical Analysis |
| 9 | Figures, run log, checksums, zip | 9 (functions), 18, 19 | Results, Reproducibility |
| 10 | Reading results and download | 20, 21 | Results, Data and Code Availability |

**How to run:** Runtime → Run all. Cell 10 promts to upload HMDD v4.0 file.This the only manual process everything else occurs programmatically.**Every cell prints a short confirmation, a table, or a plot.** The results are downloaded (`SparseNet_Results/`, zipped).

## Cell 1 — Install and import libraries
**What it does:** installs the extra packages and imports everything the analysis needs.
**Workflow step:** setup (no results produced).
**Notes:** `dynamicTreeCut` is the Python version of the R *Dynamic Tree Cut* method used to cut the gene dendrogram into modules (Step 4).
`networkx` is used only to write the miRNA–module network file.

In [ ]:
!pip -q install dynamicTreeCut networkx openpyxl

# --- Python standard library: files, text patterns, compression, hashing
import os, re, io, gzip, json, hashlib, zipfile, warnings
# --- Numerical and table tools
import numpy as np, pandas as pd
# --- Plotting
import matplotlib.pyplot as plt
# --- Downloading files from the web
import requests
# --- Statistics: linear fits, hypergeometric test, chi-square, Wilcoxon signed-rank test
from scipy import stats
from scipy.cluster.hierarchy import linkage, fcluster      # hierarchical clustering
from scipy.spatial.distance import squareform              # matrix <-> condensed distance vector
from scipy.stats import hypergeom, chi2, wilcoxon
# --- Machine-learning helpers: PCA (eigengenes, sample QC), robust covariance (outlier detection), PR/ROC metrics
from sklearn.decomposition import PCA
from sklearn.covariance import MinCovDet
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve
# --- The two classifiers used for prediction: random forest and logistic regression (plus helpers)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
# --- Dynamic tree cut (gene modules) and graph export
from dynamicTreeCut import cutreeHybrid
import networkx as nx

# --- Compatibility fix: dynamicTreeCut still calls np.in1d, which NumPy 2.x removed (np.isin is the same function).
if not hasattr(np, "in1d"):
    np.in1d = np.isin

# --- Number format: every decimal number in displayed tables, printed tables and saved CSV files shows 5 digits after the decimal point.
#     (A value smaller than 0.000005 would round to 0.00000, so those few tiny values, e.g. very small p-values, are written in scientific notation instead.)
def _fmt5(x):
    return f"{x:.5f}" if (x == 0 or abs(x) >= 5e-6 or x != x) else f"{x:.5e}"
pd.set_option("display.float_format", _fmt5)
if not getattr(pd.DataFrame.to_csv, "_five_decimals", False):      # install only once, so re-running this cell is safe
    _to_csv_orig = pd.DataFrame.to_csv
    def _to_csv_5(self, *a, **k):
        k.setdefault("float_format", _fmt5); return _to_csv_orig(self, *a, **k)
    _to_csv_5._five_decimals = True
    pd.DataFrame.to_csv = _to_csv_5

# Some correlation calculations emit harmless divide-by-zero warnings; hide them to keep the output readable.
warnings.filterwarnings("ignore", category=RuntimeWarning)
try:
    from IPython.display import display          # shows tables nicely in Colab
except ImportError:
    display = print
print("Cell 1 done: packages imported.")

## Cell 2 — Configuration (the only place you change settings)
**What it does:** defines all analysis settings in one dictionary (`CFG`) and the two cancers being compared (`CANCERS`).
**Workflow step:** Step 0 — the study design written as code.
**Key settings and what they mean:**
- `n_seeds = 30`: the whole train/validation/test procedure is repeated 30 times with different random splits (the same 30 seeds for both cancers, so results can be paired).
- `primary_ratio = 5`: for every known (positive) miRNA–disease pair, 5 *unknown* pairs are sampled as stand-in negatives. Ratios 1 and 10 are sensitivity checks. Unknown does **not** mean confirmed negative.
- `max_genes = 5000`, `iqr_min = 0.10`: gene filtering for the network.
- `min_module_size = 30`, `merge_cut = 0.25`: module size limit and the threshold for merging modules with very similar eigengenes.
- `fdr = 0.05`, `min_abs_r = 0.20`: a module counts as tumor-associated if its false-discovery-adjusted p < 0.05 **and** |correlation| ≥ 0.20.
- `learners = ["rf", "logreg"]`: the two classifiers used for prediction (random forest and logistic regression). The whole evaluation is run once for each.
- `rf_trees = 300`, `lr_C = 0.1`: random-forest size and logistic-regression regularization (fixed, not tuned on the test data).


In [ ]:
CFG = dict(
    outdir="SparseNet_Results",   # folder where every table and figure is written
    n_seeds=30,                   # matched random seeds for both cancers
    primary_ratio=5,              # unknown pairs per positive in the primary analysis
    sens_ratios=[1, 10],          # sensitivity analyses (1:1 and 10:1)
    max_genes=5000,               # keep at most this many highest-variance genes for the network
    iqr_min=0.10,                 # drop genes whose inter-quartile range is too small to be informative
    min_module_size=30,           # smallest allowed gene module
    merge_cut=0.25,               # merge modules whose eigengenes are this similar (dissimilarity < 0.25)
    fdr=0.05,                     # Benjamini-Hochberg threshold for tumor-associated modules
    min_abs_r=0.20,               # minimum |eigengene-tumor correlation|
    min_mirna_targets=5,          # miRNAs with fewer validated targets in our gene set are not tested
    learners=["rf", "logreg"],    # classifiers used for prediction: random forest and logistic regression
    rf_trees=300,                 # number of trees in the random forest
    lr_C=0.1,                     # inverse regularization strength of the (L2) logistic regression
    boot=10000,                   # bootstrap resamples for confidence intervals
    master_seed=2026,             # base random seed (seed i = master_seed + i)
)

# The two cancers compared. 'gse' = GEO expression series, 'gpl' = GEO platform (probe -> gene map),
# 'disease' = exact disease name in HMDD, 'expect' = tumor/control counts the study design requires.
CANCERS = {
    "Cholangiocarcinoma": dict(gse="GSE26566", gpl="GPL6104", disease="Cholangiocarcinoma",
                               expect=dict(tumor=104, control=59)),   # the association-SPARSE cancer
    "Breast neoplasms":   dict(gse="GSE42568", gpl="GPL570",  disease="Breast Neoplasms",
                               expect=dict(tumor=104, control=17)),   # the association-RICH cancer
}
# Known-pair counts reported in the manuscript's audit; the run warns if your HMDD file gives different numbers.
EXPECTED_HMDD_PAIRS = {"Cholangiocarcinoma": 113, "Breast Neoplasms": 524}
EXPECTED_HMDD_ENTRIES = 53530   # total evidence rows expected in HMDD v4.0 (informational)

LEARNER_NAMES = {"rf": "Random forest", "logreg": "Logistic regression"}   # display names (also used in figure titles)

print("Cell 2 done: these are the settings the analysis will use.")
display(pd.DataFrame([(k, str(v)) for k, v in CFG.items()], columns=["setting", "value"]))
display(pd.DataFrame(CANCERS).T)

## Cell 3 — Data loaders
**What it does:** defines the functions that fetch the four data sources.
**Workflow step:** Step 1 — data acquisition.
**The four sources:**
1. **Expression matrices** (GEO series matrix files): gene-activity measurements per probe per sample, plus each sample's title/description (used to decide tumor vs control).
2. **Platform annotation** (GEO platform file): tells us which gene each probe measures.
3. **miRNA targets** (OmniPath, restricted to experimentally validated miRTarBase records): which genes each miRNA is known to regulate.
4. **HMDD v4.0** (Manual upload ): the known miRNA–disease associations. These are the "positives" the models try to rank.
Downloaded files are cached in the `data/` folder, so re-running does not download them again.

In [ ]:
DATA = "data"; os.makedirs(DATA, exist_ok=True)   # local cache folder for downloads

import time, subprocess, shutil, ftplib
from urllib.parse import urlparse

def _fetch_https(url, part):
    """Route 1: Python HTTPS download, resuming from the bytes already in `part`."""
    have = os.path.getsize(part) if os.path.exists(part) else 0
    with requests.get(url, headers={"Range": f"bytes={have}-"} if have else {}, stream=True, timeout=(30, 120)) as r:
        if r.status_code == 416: return                                       # already complete
        if r.status_code not in (200, 206): raise RuntimeError(f"HTTP {r.status_code}")
        resume = have > 0 and r.status_code == 206                            # 200 means the server ignored the resume request
        total = int(r.headers.get("Content-Length", 0)) + (have if resume else 0)
        with open(part, "ab" if resume else "wb") as f:
            for chunk in r.iter_content(1 << 20): f.write(chunk)
    if total and os.path.getsize(part) != total: raise IOError(f"incomplete: got {os.path.getsize(part)} of {total} bytes")

def _fetch_wget(url, part):
    """Route 2: the `wget` program (a different network client; it can succeed when Python's is cut off). Resumes with -c."""
    if not shutil.which("wget"): raise RuntimeError("wget not available")
    subprocess.run(["wget", "-q", "-c", "-t", "3", "-T", "60", "-O", part, url], check=True)

def _fetch_ftp(url, part):
    """Route 3: plain FTP from the same NCBI server, resuming from the bytes already in `part`."""
    u = urlparse(url.replace("https://", "ftp://").replace("http://", "ftp://"))
    have = os.path.getsize(part) if os.path.exists(part) else 0
    with ftplib.FTP(u.hostname, timeout=60) as ftp:
        ftp.login(); ftp.voidcmd("TYPE I"); total = ftp.size(u.path)
        if have >= total: return
        with open(part, "ab") as f: ftp.retrbinary(f"RETR {u.path}", f.write, rest=have if have else None)
    if os.path.getsize(part) != total: raise IOError(f"incomplete: got {os.path.getsize(part)} of {total} bytes")

def download(url, path, tries=4):
    """Download `url` to `path` (skipped if already cached). If one way of downloading is cut off, the next is tried:
    Python HTTPS, then wget, then FTP. Each retries and resumes from the bytes already received, writes to a temporary
    `.part` file, and the finished file is checked (size, and gzip integrity for .gz) before it is kept."""
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    part = path + ".part"; done = False
    for route in (_fetch_https, _fetch_wget, _fetch_ftp):
        for attempt in range(1, tries + 1):
            try:
                route(url, part); done = True; break
            except Exception as e:
                print(f"  {route.__name__[7:]} attempt {attempt}/{tries} failed ({type(e).__name__}); retrying ...")
                time.sleep(min(2 * attempt, 10))
        if done: break
        print(f"  {route.__name__[7:]} did not work; trying the next route ...")
    if not done:
        raise RuntimeError(f"Could not download {url}. Open the link in your browser, save the file, upload it to the Colab "
                           f"folder '{os.path.dirname(path)}' as '{os.path.basename(path)}', and run this cell again.")
    if not os.path.exists(part) or os.path.getsize(part) < 100: raise RuntimeError(f"Download of {url} returned an empty file")
    if path.endswith(".gz"):                                                  # make sure the gzip file is complete and readable
        try:
            with gzip.open(part, "rb") as g:
                while g.read(1 << 22): pass
        except (OSError, EOFError):
            os.remove(part); raise RuntimeError(f"Downloaded file for {url} is corrupt; run this cell again.")
    os.replace(part, path)
    return path

def geo_series_url(gse):
    """Web address of a GEO series matrix file, e.g. GSE26566 -> .../GSE26nnn/GSE26566/matrix/..."""
    return f"https://ftp.ncbi.nlm.nih.gov/geo/series/{gse[:-3]}nnn/{gse}/matrix/{gse}_series_matrix.txt.gz"

def geo_platform_url(gpl):
    """Web address of a GEO platform annotation file (probe -> gene table)."""
    return f"https://ftp.ncbi.nlm.nih.gov/geo/platforms/{gpl[:-3]}nnn/{gpl}/annot/{gpl}.annot.gz"

def load_expression(gse):
    """Return (expression table: probes x samples, sample info table with each sample's title and description text)."""
    path = download(geo_series_url(gse), f"{DATA}/{gse}_series_matrix.txt.gz")
    with gzip.open(path, "rt", encoding="latin1") as f:
        lines = f.read().splitlines()
    # Header lines starting with '!Sample_' describe each sample; collect the ones we need.
    acc = title = None; chars = []; src = []
    for ln in lines:
        if ln.startswith("!Sample_geo_accession"): acc = [x.strip('"') for x in ln.split("\t")[1:]]
        elif ln.startswith("!Sample_title"): title = [x.strip('"') for x in ln.split("\t")[1:]]
        elif ln.startswith("!Sample_characteristics_ch1"): chars.append([x.strip('"') for x in ln.split("\t")[1:]])
        elif ln.startswith("!Sample_source_name_ch1"): src = [x.strip('"') for x in ln.split("\t")[1:]]
    meta = pd.DataFrame({"title": title}, index=acc)
    meta["chars"] = [" | ".join(c[i] for c in chars) + (" | " + src[i] if src else "") for i in range(len(acc))]
    # The numeric table sits between two marker lines in the file.
    b = lines.index("!series_matrix_table_begin"); e = lines.index("!series_matrix_table_end")
    tbl = pd.read_csv(io.StringIO("\n".join(lines[b + 1:e])), sep="\t", index_col=0)
    tbl.columns = [c.strip('"') for c in tbl.columns]
    return tbl, meta

def load_probe2gene(gpl):
    """Return a dictionary {probe ID: gene symbol} from the GEO platform annotation file."""
    path = download(geo_platform_url(gpl), f"{DATA}/{gpl}.annot.gz")
    with gzip.open(path, "rt", encoding="latin1") as f:
        lines = f.read().splitlines()
    b = [i for i, l in enumerate(lines) if l.startswith("!platform_table_begin")][0]
    e = [i for i, l in enumerate(lines) if l.startswith("!platform_table_end")][0]
    ann = pd.read_csv(io.StringIO("\n".join(lines[b + 1:e])), sep="\t", low_memory=False)
    col = [c for c in ann.columns if c.strip().lower() == "gene symbol"][0]
    sym = ann[col].astype(str).str.split("///").str[0].str.strip()      # if a probe lists several genes, keep the first
    ok = sym.notna() & ~sym.isin(["", "nan", "---", "NaN"])              # drop probes with no gene
    return dict(zip(ann.loc[ok, "ID"].astype(str), sym[ok]))

def load_targets():
    """Experimentally supported miRNA->gene pairs from OmniPath, keeping only records that come from miRTarBase."""
    url = "https://omnipathdb.org/interactions?datasets=mirnatarget&fields=sources&genesymbols=yes&license=academic"
    path = download(url, f"{DATA}/omnipath_mirnatarget.tsv")
    df = pd.read_csv(path, sep="\t")
    need = {"source_genesymbol", "target_genesymbol", "sources"}
    if not need.issubset(df.columns):
        raise RuntimeError(f"OmniPath columns changed: {df.columns.tolist()}")
    df = df[df["sources"].astype(str).str.contains("miRTarBase", case=False)]
    return df.rename(columns={"source_genesymbol": "mirna", "target_genesymbol": "gene"})[["mirna", "gene"]]

def load_hmdd():
    """Read the HMDD v4.0 file you upload (xlsx/csv/tsv). It needs one miRNA column and one disease column."""
    path = os.environ.get("HMDD_FILE", "")
    if not path or not os.path.exists(path):
        try:
            from google.colab import files               # Colab shows an upload button here
            print("Upload the HMDD v4.0 association file (xlsx/csv/tsv)...")
            up = files.upload(); path = list(up.keys())[0]
        except ImportError:
            raise RuntimeError("Set the HMDD_FILE environment variable to your HMDD v4.0 file.")
    if path.lower().endswith((".xlsx", ".xls")): df = pd.read_excel(path)
    else: df = pd.read_csv(path, sep=None, engine="python")
    # Find the miRNA and disease columns regardless of exact header spelling.
    cols = {c.lower().strip(): c for c in df.columns}
    mir = next(cols[c] for c in cols if c in ("mir", "mirna", "mirna_name", "mirna name"))
    dis = next(cols[c] for c in cols if c in ("disease", "disease_name", "disease name"))
    return df[[mir, dis]].rename(columns={mir: "mir", dis: "disease"})

# One dictionary so the main routine (Cell 9) can call every loader by name.
LOADERS = dict(expr=load_expression, probe2gene=load_probe2gene, targets=load_targets, hmdd=load_hmdd)

print("Cell 3 done: data loaders are defined. Nothing has been downloaded yet; downloads start in Cells 10-13.")
print("Loaders ready:", ", ".join(LOADERS))

## Cell 4 — miRNA name standardization, HMDD audit, sample labels
**What it does:** three small helpers.
- `mirna_family`: makes miRNA names comparable across databases.HMDD and OmniPath spell names differently (`hsa-miR-21-5p`, `hsa-mir-21`), so every name is reduced to a family key (`mir-21`). Without this, the targets table and the disease table would not match.
- `build_hmdd`: counts unique human miRNA–disease pairs (the **audit** that gives 113 vs 524) and builds the **association matrix** `H` (rows = miRNA families, columns = diseases, 1 = known association). `H` is the "graph" the baseline model learns from.
- `label_samples`: decides from each sample's text whether it is tumor (1) or control (0). If the counts do not match the study design, the run stops instead of guessing.
**Workflow step:** Step 2 — data audit and labeling. **Manuscript:** Authenticated Data Audit.

In [ ]:
def mirna_family(name):
    """Reduce a miRNA name to a family key so different databases match.
    hsa-miR-21-5p -> mir-21 ; hsa-mir-123-1 -> mir-123 ; hsa-miR-34a -> mir-34a ; let-7a-1 -> let-7a"""
    s = str(name).strip().lower()
    s = re.sub(r"^hsa-", "", s).replace("*", "")                    # remove species prefix and star marks
    s = re.sub(r"-(3p|5p)$", "", s)                                  # remove the mature-arm suffix
    s = re.sub(r"^(mir-\d+[a-z]?|let-7[a-z])-\d+$", r"\1", s)        # remove precursor locus number (mir-123-1 -> mir-123)
    return s

def build_hmdd(df, diseases):
    """Audit HMDD and build the miRNA x disease association matrix.
    Returns (H, audit): H has 1 where an miRNA family is associated with a disease; audit holds the pair counts."""
    d = df.dropna().copy()
    n_entries = len(d)
    d = d[d["mir"].astype(str).str.lower().str.startswith("hsa-")]   # keep human miRNAs only
    d["disease"] = d["disease"].astype(str).str.strip()
    pairs = d.drop_duplicates(["mir", "disease"])                     # one row per unique miRNA-disease pair
    audit = {"entries_total": n_entries, "human_pairs_total": len(pairs)}
    for name in diseases:
        audit[f"pairs_{name}"] = int((pairs["disease"].str.lower() == name.lower()).sum())
    d["fam"] = d["mir"].map(mirna_family)                             # harmonize names to family level
    fam = d.drop_duplicates(["fam", "disease"])
    H = pd.crosstab(fam["fam"], fam["disease"]).clip(upper=1)         # binary association matrix
    for name in diseases:
        match = [c for c in H.columns if c.lower() == name.lower()]
        audit[f"family_pairs_{name}"] = int(H[match[0]].sum()) if match else 0
    return H, audit

def label_samples(meta, gse):
    """Return a Series: 1 = tumor, 0 = control, NaN = excluded. The wording rules are specific to each GEO series."""
    txt = (meta["title"].astype(str) + " | " + meta["chars"].astype(str)).str.lower()
    y = pd.Series(np.nan, index=meta.index)
    if gse == "GSE26566":      # cholangiocarcinoma: tumors vs surrounding liver; the 6 normal bile-duct samples stay NaN (excluded)
        y[txt.str.contains("tumor") & txt.str.contains("cholangio")] = 1
        y[txt.str.contains("surrounding")] = 0
    elif gse == "GSE42568":    # breast: anything described as normal is a control, the rest are tumors
        normal = txt.str.contains("normal")
        y[~normal] = 1; y[normal] = 0
    else:
        raise ValueError(gse)
    return y

print("Cell 4 done: name and label helpers are defined. Example of how miRNA names are standardized:")
display(pd.DataFrame({"name as written": ["hsa-miR-21-5p", "hsa-mir-21", "hsa-mir-123-1", "hsa-miR-34a", "hsa-let-7a-1"]}).assign(
    family_key=lambda d: d["name as written"].map(mirna_family)))

## Cell 5 — Expression preparation, sample QC, co-expression network, modules, module–trait link
**What it does:** the core of the biology. In order:
1. `prepare_expression` — convert values to numbers, apply log2 if the data are on a raw scale, quantile-normalize so every sample has the same distribution, map probes to gene symbols (several probes for one gene → their median), drop uninformative/low-expression genes, keep the 5,000 most variable genes. *(Step 3)*
2. `sample_qc` — remove a sample only if it is an outlier by **both** network connectivity and a robust PCA test (a deliberately strict rule so real biology is not thrown away). *(Step 3)*
3. `soft_threshold` — choose the power β that makes the network approximately scale-free (few hubs, many weakly connected genes), as in WGCNA. If no power satisfies the criteria, the best available is used and flagged as a **fallback** in the log. *(Step 4)*
4. `build_network` — gene–gene similarity → soft adjacency → topological overlap (TOM) → hierarchical clustering → dynamic tree cut → merge similar modules. Each module is summarized by its **eigengene** (its main pattern across samples), which is correlated with tumor status; p-values are corrected with Benjamini-Hochberg (`bh`). Modules passing the thresholds are "selected". *(Steps 4–5)*
**Output:** for each cancer, a gene→module assignment and a table of module–tumor correlations (these become Results Tables/Figures on modules).

In [ ]:
def quantile_normalize(df):
    """Make every sample (column) have the same distribution of values by averaging the sorted values across samples."""
    arr = df.values; n, m = arr.shape
    order = np.argsort(arr, axis=0)                                   # rank order of genes within each sample
    mean_sorted = np.take_along_axis(arr, order, axis=0).mean(axis=1) # average value at each rank
    ranks = np.empty_like(order)
    np.put_along_axis(ranks, order, np.tile(np.arange(n)[:, None], (1, m)), axis=0)
    return pd.DataFrame(mean_sorted[ranks], index=df.index, columns=df.columns)

def prepare_expression(raw, probe2gene, cfg):
    """Raw probe x sample table -> clean gene x sample table (+ a dictionary of counts for the log)."""
    X = raw.apply(pd.to_numeric, errors="coerce")
    miss = float(X.isna().values.mean())                              # fraction of missing values
    if miss > 0.01: raise RuntimeError(f"Missingness {miss:.3%} exceeds 1%")
    X = X.apply(lambda c: c.fillna(c.median()), axis=0)               # fill the few gaps with the sample median
    logged = False
    if X.values.max() > 100:                                          # raw-scale intensities -> log2 scale
        X = np.log2(X.clip(lower=0) + 1); logged = True
    X = quantile_normalize(X)
    g = pd.Series(X.index.astype(str)).map(probe2gene)                # probe -> gene symbol
    X = X[g.notna().values]; X.index = g.dropna().values
    n_mapped = X.shape[0]
    X = X.groupby(level=0).median()                                   # one row per gene (median of its probes)
    iqr = X.quantile(.75, axis=1) - X.quantile(.25, axis=1)           # spread of each gene across samples
    mean = X.mean(axis=1)
    X = X[(iqr > cfg["iqr_min"]) & (mean >= np.percentile(mean, 20))] # drop flat genes and the lowest-expressed 20%
    n_after_filter = X.shape[0]
    X = X.loc[X.var(axis=1).sort_values(ascending=False).index[:cfg["max_genes"]]]  # keep the most variable genes
    return X, dict(missing_fraction=miss, log2_applied=logged, probes_mapped=n_mapped,
                   genes_after_collapse=int(len(mean)), genes_after_filter=int(n_after_filter), genes_used=int(X.shape[0]))

def sample_qc(X):
    """Find outlier samples. A sample is removed only if BOTH tests flag it:
    (1) its network connectivity Z-score < -3, and (2) its squared robust distance in 5-PC space exceeds the 99.9% chi-square cutoff."""
    S = (1 + np.corrcoef(X.values.T)) / 2                             # sample-sample similarity
    np.fill_diagonal(S, 0); k = S.sum(1)                              # connectivity of each sample
    Zc = (k - k.mean()) / k.std(ddof=1)
    pcs = PCA(n_components=5).fit_transform((X.values - X.values.mean(1, keepdims=True)).T)
    d2 = MinCovDet(random_state=0).fit(pcs).mahalanobis(pcs)          # robust distance of each sample
    out = (Zc < -3) & (d2 > chi2.ppf(0.999, 5))
    return X.columns[~out], dict(removed=list(X.columns[out]))

def soft_threshold(S, r2_min=0.80, kmin=5):
    """Pick the soft-threshold power beta. Try powers 1..30; choose the first giving a scale-free network
    (signed R^2 >= 0.80, negative slope, mean connectivity >= 5). Returns (beta, used_fallback, table of all powers)."""
    rows = []
    for b in list(range(1, 11)) + list(range(12, 31, 2)):
        A = S ** b; np.fill_diagonal(A, 0); k = A.sum(1)               # adjacency at this power and gene connectivity
        lk = np.log10(k + 1e-12); bins = pd.cut(lk, 10)
        t = pd.DataFrame({"lk": lk, "bin": bins}).groupby("bin", observed=True).agg(n=("lk", "size"), lk=("lk", "mean"))
        t = t[t.n > 0]; p = np.log10(t.n / t.n.sum())
        slope, icpt, r, _, _ = stats.linregress(t.lk, p) if len(t) > 2 else (0, 0, 0, 0, 0)  # scale-free fit: log p(k) ~ log k
        rows.append(dict(power=b, slope=slope, r2=r * r, signed_r2=-np.sign(slope) * r * r, mean_k=k.mean()))
    tab = pd.DataFrame(rows)
    ok = tab[(tab.signed_r2 >= r2_min) & (tab.slope < 0) & (tab.mean_k >= kmin)]
    if len(ok): return int(ok.power.iloc[0]), False, tab
    neg = tab[tab.slope < 0]                                          # fallback: best signed R^2 among negative slopes
    return int((neg if len(neg) else tab).sort_values("signed_r2", ascending=False).power.iloc[0]), True, tab

def eigengene(Xm):
    """Summarize a module by its eigengene: the first principal component of its standardized expression,
    sign-flipped so it points the same way as the module's average expression."""
    Z = ((Xm.values - Xm.values.mean(1, keepdims=True)) / (Xm.values.std(1, keepdims=True) + 1e-12)).T
    pc = PCA(n_components=1).fit_transform(Z)[:, 0]
    if np.corrcoef(pc, Z.mean(1))[0, 1] < 0: pc = -pc
    return pc

def bh(p):
    """Benjamini-Hochberg false-discovery-rate adjustment of a list of p-values (returns q-values)."""
    p = np.asarray(p, float); n = len(p); o = np.argsort(p)
    q = np.empty(n); prev = 1.0
    for rank, i in zip(range(n, 0, -1), o[::-1]):
        prev = min(prev, p[i] * n / rank); q[i] = prev
    return q

def build_network(X, y, cfg):
    """Gene x sample table + tumor labels -> co-expression network, gene modules, and module-tumor correlations."""
    genes = X.index.values
    S = (1 + np.corrcoef(X.values)) / 2                               # signed similarity between genes, scaled to 0..1
    beta, fallback, ptab = soft_threshold(S)                          # choose the soft-threshold power
    A = S ** beta; np.fill_diagonal(A, 0)                             # soft adjacency (strong links emphasized)
    k = A.sum(1); L = A @ A                                           # connectivity; shared-neighbour sums
    T = (L + A) / (np.minimum.outer(k, k) + 1 - A); np.fill_diagonal(T, 1)   # topological overlap matrix (TOM)
    D = 1 - T; np.fill_diagonal(D, 0); D = (D + D.T) / 2              # dissimilarity = 1 - TOM
    Zl = linkage(squareform(D, checks=False), method="average")       # hierarchical clustering tree (dendrogram)
    res = cutreeHybrid(Zl, D, minClusterSize=cfg["min_module_size"], deepSplit=2)  # cut the tree into modules
    lab = np.asarray(res["labels"]).astype(int)                       # module label per gene (0 = unassigned/"grey")
    n_initial = len(set(lab) - {0})
    # --- merge modules whose eigengenes are too similar (dissimilarity < merge_cut)
    mods = sorted(set(lab) - {0})
    ME = pd.DataFrame({m: eigengene(X.loc[genes[lab == m]]) for m in mods}, index=X.columns)
    if len(mods) > 1:
        Dm = 1 - ME.corr().values; np.fill_diagonal(Dm, 0)
        cl = fcluster(linkage(squareform((Dm + Dm.T) / 2, checks=False), "average"), cfg["merge_cut"], "distance")
        remap = dict(zip(mods, cl)); lab = np.array([remap.get(l, 0) for l in lab])
    # --- renumber modules so that module 1 is the largest
    sizes = pd.Series(lab[lab > 0]).value_counts()
    rename = {old: i + 1 for i, old in enumerate(sizes.index)}
    lab = np.array([rename.get(l, 0) for l in lab])
    mods = sorted(set(lab) - {0})
    ME = pd.DataFrame({m: eigengene(X.loc[genes[lab == m]]) for m in mods}, index=X.columns)
    # --- module-trait association: correlate each eigengene with tumor status (1/0)
    n = len(y); r = np.array([np.corrcoef(ME[m].values, y.values)[0, 1] for m in mods])
    t = r * np.sqrt((n - 2) / (1 - r ** 2)); p = 2 * stats.t.sf(np.abs(t), n - 2)
    q = bh(p)                                                         # correct for testing many modules
    mt = pd.DataFrame(dict(module=mods, size=[int((lab == m).sum()) for m in mods], r=r, p=p, fdr=q))
    sel = mt[(mt.fdr < cfg["fdr"]) & (mt.r.abs() >= cfg["min_abs_r"])]   # tumor-associated modules
    fb = False
    if sel.empty:                                                     # exploratory fallback: two smallest-FDR modules
        sel = mt.sort_values("fdr").head(2); fb = True
    return dict(genes=genes, labels=lab, beta=beta, beta_fallback=fallback, power_table=ptab, ME=ME,
                module_trait=mt, selected=sel, selected_is_fallback=fb,
                n_initial_modules=n_initial, n_modules=len(mods), n_grey=int((lab == 0).sum()), linkage=Zl,
                adj_head=A[:50, :50].copy(), diss_head=D[:50, :50].copy())   # 50-gene corners, used only for plots

print("Cell 5 done: functions ready: prepare_expression, sample_qc, soft_threshold, build_network (network, modules, module-trait).")

## Cell 6 — miRNA–module enrichment and the continuous prior
**What it does:** for each tumor-associated module and each miRNA, asks: *are this miRNA's validated target genes over-represented in this module?*
- `module_enrichment` uses the **hypergeometric test** (the probability of seeing at least `x` of the miRNA's `n` targets inside a module of `K` genes drawn from `N` genes by chance), then corrects all p-values with Benjamini-Hochberg (`q`). The enrichment **score** is `-log10(q) × |module–tumor correlation|`: bigger when the overlap is more surprising and the module tracks tumor status more strongly.
- `make_prior` keeps, for every miRNA, its **best score across modules**. miRNAs with no enrichment get 0. This vector is the **prior** `P(m, d)`.
**Workflow step:** Step 6. **Important:** the prior uses only expression and target data. It never sees HMDD labels, which is why it cannot leak the answer.
**Reading the output:** `n_enrichments_q05` in the run log counts enrichments with q < 0.05. If it is 0, the prior is built from non-significant signals and should not be read as evidence of specific regulators.

In [ ]:
def module_enrichment(net, targets, expr_genes, cfg):
    """Hypergeometric test of every miRNA family against every selected module. Returns a table sorted by q-value."""
    tg = targets.copy(); tg["fam"] = tg["mirna"].map(mirna_family)    # standardize miRNA names
    universe = set(expr_genes) & set(tg["gene"])                      # genes that are both measured and have a known miRNA
    tg = tg[tg["gene"].isin(universe)].drop_duplicates(["fam", "gene"])
    N = len(universe); rows = []
    mir_sets = {f: set(g["gene"]) for f, g in tg.groupby("fam")}      # target gene set for each miRNA family
    gene_arr = np.asarray(net["genes"])
    for _, m in net["selected"].iterrows():                           # loop over tumor-associated modules
        mod_genes = set(gene_arr[net["labels"] == m.module]) & universe
        K = len(mod_genes)                                            # module size within the universe
        for fam, tset in mir_sets.items():
            n = len(tset)                                             # number of this miRNA's targets
            if n < cfg["min_mirna_targets"]: continue                 # skip miRNAs with too few targets
            x = len(tset & mod_genes)                                 # targets that fall inside the module
            if x == 0: continue
            rows.append(dict(module=int(m.module), module_r=m.r, mirna=fam, overlap=x, mirna_targets=n,
                             module_genes_in_universe=K, universe=N, p=hypergeom.sf(x - 1, N, K, n)))
    enr = pd.DataFrame(rows)
    if enr.empty: return enr
    enr["q"] = bh(enr["p"].values)                                    # correct for all miRNA-module tests
    enr["score"] = -np.log10(np.maximum(enr["q"], 1e-300)) * enr["module_r"].abs()
    return enr.sort_values("q").reset_index(drop=True)

def make_prior(enr, mirna_index):
    """One prior value per miRNA family in HMDD: its best enrichment score (0 if never enriched)."""
    if enr.empty: return pd.Series(0.0, index=mirna_index)
    return enr.groupby("mirna")["score"].max().reindex(mirna_index).fillna(0.0)

print("Cell 6 done: functions ready: module_enrichment, make_prior.")

## Cell 7 — Positive–unlabelled evaluation (does the prior help?)
**What it does:** the fair test of the prior, run **once for each classifier** (random forest and logistic regression). For each cancer and each of 30 random seeds:
1. **Split** the known positive miRNAs: 20% test, the remainder training. Sample *unknown* miRNAs as stand-in negatives (5 per positive in the primary analysis): one set for the test, and a separate, non-overlapping set for training. Unknown does **not** mean confirmed negative.
2. **Features:** each miRNA is described by its row of the association matrix `H` **with the target disease's column removed** (its associations with all other diseases), so held-out labels cannot leak in.
3. **Three models per classifier, same split and same test pairs:**
   - `baseline`: classifier on the association profile only (no prior);
   - `full`: the same profile **plus the co-expression prior** as one extra feature (**AUPRC(prior)**);
   - `prior_only` (ablation): the prior as the only feature.
4. **ΔAUPRC for that seed** = AUPRC(prior) − AUPRC(baseline) = AUPRC(`full`) − AUPRC(`baseline`). The sign is always prior minus baseline, so ΔAUPRC > 0 means the prior helped. This is the main quantity the notebook reports, separately for each classifier.
**Classifiers:** `rf` = random forest (300 trees, balanced class weights); `logreg` = L2-regularized logistic regression on standardized features (balanced class weights, C = 0.1). Settings are fixed in `CFG`; nothing is tuned on the test set.
**Metric:** AUPRC (area under the precision–recall curve) is primary because positives are rare. With 5 unknowns per positive, chance AUPRC ≈ 1/6 ≈ 0.17.
**Workflow step:** Step 7. **Manuscript:** Experimental Design, Leakage-Prevention Strategy.

In [ ]:
def make_model(kind, cfg, seed):
    """Build an (untrained) classifier. 'rf' = random forest; 'logreg' = logistic regression on standardized features."""
    if kind == "rf":
        return RandomForestClassifier(n_estimators=cfg["rf_trees"], min_samples_leaf=2, class_weight="balanced_subsample",
                                      n_jobs=-1, random_state=seed)
    if kind == "logreg":
        return make_pipeline(StandardScaler(), LogisticRegression(C=cfg["lr_C"], class_weight="balanced", max_iter=5000))
    raise ValueError(kind)

def feature_sets(H, disease, prior):
    """Feature matrices (one row per miRNA family) for the three models. The target disease's own column is removed first (prevents leakage)."""
    Z = H.drop(columns=[disease]).values.astype(float)               # associations with all OTHER diseases
    P = prior.values.astype(float)[:, None]                          # co-expression prior (built from expression + targets only)
    return dict(baseline=Z, prior_only=P, full=np.hstack([Z, P]))

PR_GRID = np.linspace(0, 1, 101)                                     # recall values at which precision is reported

def pr_on_grid(labels, scores):
    """Precision-recall curve of ONE test set, as interpolated precision at every recall in PR_GRID
    (precision at recall r = the highest precision reached at any recall >= r). This puts every seed's curve on the same recall axis
    so that curves can be averaged across the 30 seeds."""
    p, r, _ = precision_recall_curve(labels, scores)
    return np.array([p[r >= g].max() for g in PR_GRID])

def best_threshold(scores, labels):
    """Score cut-off that maximizes F1 on a labelled set (only needed if precision/recall/F1 are reported)."""
    p, r, th = precision_recall_curve(labels, scores)
    f = 2 * p[:-1] * r[:-1] / np.maximum(p[:-1] + r[:-1], 1e-12)
    return th[np.argmax(f)] if len(th) else 0.5

def prf(scores, labels, thr):
    """Precision, recall and F1 when predicting 'positive' for every score >= thr."""
    pred = scores >= thr; tp = int((pred & (labels == 1)).sum())
    P = tp / max(pred.sum(), 1); R = tp / max(labels.sum(), 1)
    return P, R, (2 * P * R / (P + R) if P + R > 0 else 0.0)

def one_split(H, disease, feats, seed, ratio, cfg, learner):
    """One random train/test split for one disease. Trains the classifier on training positives + training unknowns and returns the
    test AUPRC of `baseline` (profile only), `prior_only` (ablation) and `full` (profile + prior)."""
    rng = np.random.default_rng(seed)
    idx_all = np.arange(len(H)); y = H[disease].values == 1
    pos = rng.permutation(idx_all[y]); unk = rng.permutation(idx_all[~y])           # shuffled known positives / unknown pairs
    n_te = max(1, round(.2 * len(pos))); te_p, tr_p = pos[:n_te], pos[n_te:]         # 20% of positives -> test, the rest -> training
    n_te_u = min(ratio * len(te_p), len(unk) // 2)
    te_u = unk[:n_te_u]                                                              # unknowns for the test set
    tr_u = unk[n_te_u:n_te_u + ratio * len(tr_p)]                                    # unknowns for training (no overlap with the test unknowns)
    tr_idx = np.r_[tr_p, tr_u]; tr_y = np.r_[np.ones(len(tr_p)), np.zeros(len(tr_u))]
    te_idx = np.r_[te_p, te_u]; te_y = np.r_[np.ones(len(te_p)), np.zeros(len(te_u))]
    assert not (set(tr_idx) & set(te_idx)), "training and test miRNAs overlap"      # safety check: no miRNA is both trained on and tested
    assert te_y.sum() > 0 and (te_y == 0).sum() > 0, "test set needs both positives and unknowns"
    out = {"_pr": {}}
    for tag, F in feats.items():
        mdl = make_model(learner, cfg, seed).fit(F[tr_idx], tr_y)                    # train on training pairs only
        s = mdl.predict_proba(F[te_idx])[:, 1]                                       # predicted probability of "associated", test miRNAs only
        out[tag] = dict(auprc=average_precision_score(te_y, s))                      # AUPRC measured on the TEST set
        out["_pr"][tag] = pr_on_grid(te_y, s)                                        # precision-recall curve of this test set
    out["_n"] = dict(train_pos=len(tr_p), train_unk=len(tr_u), test_pos=len(te_p), test_unk=len(te_u))
    return out

def evaluate(H, disease, prior, cfg, ratio, learner):
    """Repeat one_split for all seeds. Returns (table with one row per seed and model, table of precision-recall curves per seed and model)."""
    feats = feature_sets(H, disease, prior); rows, crv = [], []
    for s in range(cfg["n_seeds"]):
        res = one_split(H, disease, feats, cfg["master_seed"] + s, ratio, cfg, learner)
        for tag, v in res.items():
            if tag in ("_n", "_pr"): continue
            rows.append(dict(seed=s, ratio=ratio, learner=learner, model=tag, **v, **res["_n"]))
        for tag, pr in res["_pr"].items():
            crv.append(pd.DataFrame(dict(seed=s, ratio=ratio, learner=learner, model=tag, recall=PR_GRID, precision=pr)))
    return pd.DataFrame(rows), pd.concat(crv, ignore_index=True)

def rank_candidates(H, disease, prior, cfg, learner, top=20):
    """After evaluation is finished: rank not-yet-associated miRNAs with the `full` model (profile + prior). Scores come from 5-fold
    cross-validation, so no miRNA is scored by a model that was trained on it. Leads for follow-up, not confirmed regulators."""
    F = feature_sets(H, disease, prior)["full"]; y = H[disease].values == 1
    cv = StratifiedKFold(5, shuffle=True, random_state=cfg["master_seed"])
    S = cross_val_predict(make_model(learner, cfg, cfg["master_seed"]), F, y, cv=cv, method="predict_proba")[:, 1]
    df = pd.DataFrame(dict(mirna=H.index, score=S, prior=prior.values, known=y))
    return df[~df.known].sort_values("score", ascending=False).head(top).reset_index(drop=True)

print("Cell 7 done: functions ready: make_model, feature_sets, one_split, evaluate (AUPRC and precision-recall curves), rank_candidates.")

## Cell 8 — Statistics
**What it does:** turns the 30 per-seed results into the numbers.
- `boot_ci`: mean and 95% **bootstrap confidence interval** (resample the 30 seed results 10,000 times).
- `paired_stats`: for each cancer, **ΔAUPRC = AUPRC(prior) − AUPRC(baseline)** for each seed, its mean with a 95% bootstrap CI, and a one-sided paired **Wilcoxon signed-rank test** that ΔAUPRC > 0.
**Workflow step:** Step 8. **Manuscript:** Evaluation Metrics and Statistical Analysis; Results.

In [ ]:
def boot_ci(x, n=10000, seed=1):
    """Mean and 95% bootstrap confidence interval of a list of per-seed values."""
    x = np.asarray(x, float); rng = np.random.default_rng(seed)
    m = rng.choice(x, (n, len(x)), replace=True).mean(1)
    return x.mean(), np.percentile(m, 2.5), np.percentile(m, 97.5)

def paired_stats(ev, cfg, cancers):
    """ev maps cancer -> per-seed results. Returns (table of delta-AUPRC per cancer, dict of per-seed delta-AUPRC)."""
    delta, out = {}, []
    for c, df in ev.items():
        auprc_baseline = df[df.model == "baseline"].sort_values("seed")["auprc"].values   # AUPRC without the prior, one value per seed
        auprc_prior = df[df.model == "full"].sort_values("seed")["auprc"].values          # AUPRC with the prior, one value per seed
        delta[c] = d = auprc_prior - auprc_baseline                             # delta AUPRC = AUPRC(prior) - AUPRC(baseline), per seed; same formula for every cancer
        m, lo, hi = boot_ci(d, cfg["boot"])
        try: pv = wilcoxon(d, alternative="greater").pvalue                     # one-sided paired Wilcoxon: is delta > 0 ?
        except ValueError: pv = np.nan
        out.append(dict(cancer=c, delta_auprc=m, ci_low=lo, ci_high=hi, p_one_sided=pv,
                        auprc_baseline=auprc_baseline.mean(), auprc_prior=auprc_prior.mean(), seeds_improved=int((d > 0).sum()), n_seeds=len(d)))
    return pd.DataFrame(out), delta

ABL_ORDER = ["baseline", "prior_only", "full"]    # order of the ablation arms in tables and figure

def _paired(a, b, boot):
    """Mean paired difference a - b over seeds with 95% bootstrap CI and one-sided Wilcoxon p (is a > b ?)."""
    d = np.asarray(a, float) - np.asarray(b, float); m, lo, hi = boot_ci(d, boot)
    try: pv = wilcoxon(d, alternative="greater").pvalue
    except ValueError: pv = np.nan
    return m, lo, hi, pv

def ablation_stats(ev, cfg):
    """Ablation table: mean test AUPRC (95% bootstrap CI) of each arm per cancer, and each arm's paired difference
    to the baseline (arm minus baseline, with CI and one-sided Wilcoxon p)."""
    rows = []
    for c, df in ev.items():
        wide = df.pivot(index="seed", columns="model", values="auprc")
        for mdl in [m for m in ABL_ORDER if m in wide]:
            m, lo, hi = boot_ci(wide[mdl].values, cfg["boot"])
            row = dict(cancer=c, model=mdl, auprc=m, ci_low=lo, ci_high=hi, n_seeds=len(wide))
            if mdl != "baseline":
                dm, dlo, dhi, pv = _paired(wide[mdl].values, wide["baseline"].values, cfg["boot"])
                row.update(delta_vs_baseline=dm, delta_vs_baseline_ci_low=dlo, delta_vs_baseline_ci_high=dhi, p_vs_baseline=pv)
            rows.append(row)
    return pd.DataFrame(rows)

print("Cell 8 done: statistics functions ready (bootstrap CI, the delta-AUPRC test and the ablation table).")

## Cell 9 — Output helpers (tables, figures, zip) and the one-command alternative
**What it does:** define functions that write result tables (`save_tables`), draw figures (`save_figures`, `qc_figures`, `network_figures`, `enrichment_figure`), and package the zip (`package_results`). Every figure is **saved and also shown** in the notebook (`SHOW_PLOTS = True`; set it to `False` to only save). It also defines `run_all`, which performs Cells 10 to 19 in a single call.**
**Workflow step:** Results output.



In [ ]:
PAL = dict(blue="#2F6690", orange="#C8553D", gray="#8A8A8A", green="#3A7D44")   # figure colors
SHOW_PLOTS = True                                                                # show every figure in the notebook as well as saving it

def _save(path):
    """Save the current figure to `path`, show it in the notebook, then close it."""
    plt.savefig(path, bbox_inches="tight", pad_inches=0.15)      # tight box: axis labels and titles are never cut off
    if SHOW_PLOTS: plt.show()
    plt.close()

def sha256(path):
    """Checksum of a file (lets anyone verify the outputs were not altered)."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for ch in iter(lambda: f.read(1 << 20), b""): h.update(ch)
    return h.hexdigest()

def audit_associations(L, cancers, expected_pairs, log):
    """STEP 2: load HMDD, print the audit, warn if the pair counts differ from the manuscript. Returns (H, audit)."""
    H, audit = build_hmdd(L["hmdd"](), [v["disease"] for v in cancers.values()])
    log["hmdd_audit"] = audit; print("HMDD audit:", audit)
    for d, n in (expected_pairs or {}).items():
        if audit.get(f"pairs_{d}") != n: print(f"WARNING: HMDD pairs for {d} = {audit.get(f'pairs_{d}')}, manuscript says {n}. Check your HMDD snapshot.")
    return H, audit

def prepare_cancer_arm(name, c, L, targets, H, cfg, out):
    """STEPS 3-6 for ONE cancer: expression -> QC -> network/modules -> enrichment -> prior.
    Returns (info for the log, network, enrichment table, prior)."""
    print(f"\n=== {name} ({c['gse']}) ===")
    raw, meta = L["expr"](c["gse"]); y = label_samples(meta, c["gse"])
    counts = dict(tumor=int((y == 1).sum()), control=int((y == 0).sum()), excluded=int(y.isna().sum()))
    print("sample labels:", counts)
    if (counts["tumor"], counts["control"]) != (c["expect"]["tumor"], c["expect"]["control"]):   # stop if labels do not match the design
        raise RuntimeError(f"Sample labels for {c['gse']} do not match the design {c['expect']}: inspect `meta` and adjust label_samples().")
    y = y.dropna(); raw = raw[y.index]                                # keep only labelled samples
    X, info = prepare_expression(raw, L["probe2gene"](c["gpl"]), cfg)  # Step 3
    keep, qc = sample_qc(X); X = X[keep]; y = y[keep]                  # Step 3: remove outlier samples
    info.update(qc); info.update(samples_retained=int(X.shape[1]), tumor=int((y == 1).sum()), control=int((y == 0).sum()))
    print("preparation:", {k: v for k, v in info.items() if k != "removed"}, "removed:", qc["removed"])
    net = build_network(X, y, cfg)                                    # Steps 4-5: modules and module-trait correlations
    enr = module_enrichment(net, targets, X.index, cfg)               # Step 6: miRNA-module enrichment
    prior = make_prior(enr, H.index)                                  # Step 6: the prior
    info.update(beta=net["beta"], beta_fallback=net["beta_fallback"], n_modules=net["n_modules"],
                n_initial_modules=net["n_initial_modules"], n_grey_genes=net["n_grey"],
                n_significant_modules=int(0 if net["selected_is_fallback"] else len(net["selected"])),
                selected_is_fallback=net["selected_is_fallback"],
                n_enrichments_q05=int((enr["q"] < 0.05).sum()) if len(enr) else 0,
                n_mirnas_with_prior=int((prior > 0).sum()))
    print({k: v for k, v in info.items() if k != "removed"})
    save_exploratory_figures(name, c, raw, X, y, net, enr, out)       # QC/exploratory figures for this cancer
    net["module_trait"].to_csv(f"{out}/tables/ModuleTrait_{c['gse']}.csv", index=False)
    net["power_table"].to_csv(f"{out}/tables/SoftThreshold_{c['gse']}.csv", index=False)
    return info, net, enr, prior

def evaluate_all(H, cancers, priors, cfg, out, learner):
    """STEP 7: evaluate every cancer at the primary ratio and at the sensitivity ratios. Returns {(cancer, ratio): table}."""
    ev_all = {}
    for name, c in cancers.items():
        dis = [col for col in H.columns if col.lower() == c["disease"].lower()][0]   # exact disease column in H
        for ratio in [cfg["primary_ratio"]] + cfg["sens_ratios"]:
            df, crv = evaluate(H, dis, priors[name], cfg, ratio, learner); df["cancer"] = name; crv["cancer"] = name
            ev_all[(name, ratio)] = df; ev_all[("PR", name, ratio)] = crv      # AUPRC table and precision-recall curves
    pd.concat([v for k, v in ev_all.items() if k[0] != "PR"], ignore_index=True).to_csv(f"{out}/tables/Evaluation_PerSeed_AllRatios_{learner}.csv", index=False)
    pd.concat([v for k, v in ev_all.items() if k[0] == "PR"], ignore_index=True).to_csv(f"{out}/tables/PR_Curves_PerSeed_{learner}.csv", index=False)
    return ev_all

def save_tables(cfg, cancers, H, audit, ev_all, enrs, priors, out, learner):
    """STEP 8: delta-AUPRC statistics and every results table. Returns (tests, delta)."""
    names = list(cancers)
    primary = {n: ev_all[(n, cfg["primary_ratio"])] for n in names}
    tests, delta = paired_stats(primary, cfg, names); tests.insert(0, "learner", learner)
    tests.to_csv(f"{out}/tables/Delta_AUPRC_{learner}.csv", index=False)                   # the main result: delta AUPRC per cancer
    chk = (tests["auprc_prior"] - tests["auprc_baseline"] - tests["delta_auprc"]).abs().max()   # sanity check, both cancers
    assert chk < 1e-9, f"delta_auprc is not AUPRC(prior) - AUPRC(baseline) (max difference {chk})"
    print(f"[{LEARNER_NAMES[learner]}]"); print("Check passed: delta AUPRC = AUPRC(prior) - AUPRC(baseline) for", ", ".join(tests["cancer"]))
    print("\n", tests.round(5).to_string(index=False))
    sens = []                                                                    # delta AUPRC again at the other unknown-pair ratios
    for ratio in cfg["sens_ratios"]:
        t2, _ = paired_stats({n: ev_all[(n, ratio)] for n in names}, cfg, names); t2["ratio"] = ratio; sens.append(t2)
    pd.concat(sens).to_csv(f"{out}/tables/Sensitivity_Ratios_{learner}.csv", index=False)
    comp = []                                                                    # one-line summary per cancer (sparse vs rich)
    tt = tests.set_index("cancer")
    for n in names:
        row = dict(cancer=n, hmdd_pairs=audit.get(f"pairs_{cancers[n]['disease']}"), hmdd_family_pairs=audit.get(f"family_pairs_{cancers[n]['disease']}"),
                   auprc_baseline=float(tt.loc[n, "auprc_baseline"]), auprc_prior=float(tt.loc[n, "auprc_prior"]),   # mean test AUPRC over seeds, without / with the prior
                   delta_auprc=float(delta[n].mean()))
        comp.append(row)
    pd.DataFrame(comp).to_csv(f"{out}/tables/Sparse_Rich_Comparison_{learner}.csv", index=False)
    ablation_stats(primary, cfg).to_csv(f"{out}/tables/Ablation_AUPRC_{learner}.csv", index=False)   # ablation: AUPRC of every arm per cancer
    # enrichment table, network edges (only q < 0.05) and GraphML network
    allenr = pd.concat([e.assign(cancer=n) for n, e in enrs.items() if len(e)], ignore_index=True) if any(len(e) for e in enrs.values()) else pd.DataFrame()
    allenr.to_csv(f"{out}/tables/miRNA_Module_Enrichment.csv", index=False)
    sig = allenr[allenr["q"] < 0.05] if len(allenr) else allenr
    edge_cols = ["cancer", "mirna", "module", "overlap", "q"]
    (sig[edge_cols] if len(sig) else pd.DataFrame(columns=edge_cols)).rename(columns={"mirna": "Source", "module": "Target"}).to_csv(f"{out}/tables/Network_Edges.csv", index=False)
    G = nx.Graph()
    for _, r in sig.iterrows():
        G.add_node(r.mirna, kind="miRNA"); G.add_node(f"{r.cancer}:M{r.module}", kind="module")
        G.add_edge(r.mirna, f"{r.cancer}:M{r.module}", weight=float(-np.log10(max(r.q, 1e-300))), cancer=r.cancer)
    nx.write_graphml(G, f"{out}/tables/miRNA_Module_Network.graphml")
    for n in names:                                                              # post-evaluation candidate ranking
        dis = [col for col in H.columns if col.lower() == cancers[n]["disease"].lower()][0]
        rank_candidates(H, dis, priors[n], cfg, learner).to_csv(f"{out}/tables/Candidates_{n.replace(' ', '_')}_{learner}.csv", index=False)
    pr_curve_tables(cfg, cancers, ev_all, out, learner)                          # precision-recall curve table and summary
    return tests, delta

def save_figures(cfg, cancers, audit, delta, nets, out, learner):
    """Draw and save all figures as PNG files."""
    names = list(cancers)
    # Figure: number of known miRNA-disease pairs (sparse vs rich)
    fig, ax = plt.subplots(figsize=(4.4, 3.4), dpi=200)
    vals = [audit.get(f"pairs_{cancers[n]['disease']}") for n in names]
    b = ax.bar(names, vals, color=[PAL["blue"], PAL["orange"]], width=.55)
    for r_, v in zip(b, vals): ax.text(r_.get_x() + r_.get_width() / 2, v + max(vals) * .02, str(v), ha="center", fontsize=9)
    ax.set_ylabel("Unique miRNA-disease pairs"); [ax.spines[s].set_visible(False) for s in ("top", "right")]
    plt.tight_layout(); _save(f"{out}/figures/Fig_AssociationPairs.png")
    # Figure: per-seed delta AUPRC for each cancer
    fig, ax = plt.subplots(figsize=(5.2, 3.6), dpi=200)
    for i, n in enumerate(names):
        d = delta[n]; ax.scatter(np.full(len(d), i) + np.random.default_rng(0).uniform(-.12, .12, len(d)), d, s=14, color=[PAL["blue"], PAL["orange"]][i], alpha=.7)
        m, lo, hi = boot_ci(d, cfg["boot"]); ax.errorbar(i, m, yerr=[[m - lo], [hi - m]], color="black", capsize=5, lw=1.5)
    ax.axhline(0, color="#666", lw=.8); ax.set_xticks(range(len(names))); ax.set_xticklabels(names)
    ax.set_ylabel("Delta AUPRC per seed\n(AUPRC(prior) - AUPRC(baseline))", fontsize=8.5); [ax.spines[s].set_visible(False) for s in ("top", "right")]
    ax.set_title(LEARNER_NAMES[learner], fontsize=9); plt.tight_layout(); _save(f"{out}/figures/Fig_DeltaAUPRC_{learner}.png")
    # Figures per cancer: strongest module-tumor correlations and module size distribution
    for n, c in cancers.items():
        mt = nets[n]["module_trait"].copy(); mt["abs"] = mt.r.abs(); top = mt.sort_values("abs", ascending=False).head(15).sort_values("r")
        fig, ax = plt.subplots(figsize=(6, 4), dpi=200)
        ax.barh([f"Module {m}" for m in top.module], top.r, color=[PAL["orange"] if v > 0 else PAL["blue"] for v in top.r])
        ax.axvline(0, color="#444", lw=.8); ax.set_xlabel("Eigengene-tumor correlation (r)"); [ax.spines[s].set_visible(False) for s in ("top", "right")]
        plt.tight_layout(); _save(f"{out}/figures/Fig_ModuleTrait_{c['gse']}.png")
        fig, ax = plt.subplots(figsize=(6, 3.2), dpi=200); sz = mt.sort_values("size", ascending=False)["size"].values
        ax.bar(range(1, len(sz) + 1), sz, color=PAL["blue"]); ax.set_xlabel("Module (ordered by size)"); ax.set_ylabel("Genes")
        [ax.spines[s].set_visible(False) for s in ("top", "right")]; plt.tight_layout(); _save(f"{out}/figures/Fig_ModuleSizes_{c['gse']}.png")

def pr_curve_tables(cfg, cancers, ev_all, out, learner):
    """Precision-recall tables (primary analysis). PR_Curve_<learner>.csv: mean, SD and 95% range over seeds of precision at each recall.
    PR_Summary_<learner>.csv: AUPRC and precision at recall 0.2 / 0.4 / 0.6 / 0.8 per cancer and model, next to the chance level."""
    curve, summ = [], []
    for n in cancers:
        crv = ev_all[("PR", n, cfg["primary_ratio"])]; df = ev_all[(n, cfg["primary_ratio"])]
        chance = float((df.test_pos / (df.test_pos + df.test_unk)).mean())      # precision expected from random scores = share of positives in the test set
        for mdl in [m for m in ABL_ORDER if m in set(crv.model)]:
            w = crv[crv.model == mdl].pivot(index="seed", columns="recall", values="precision")   # seeds x recall grid
            curve.append(pd.DataFrame(dict(learner=learner, cancer=n, model=mdl, recall=w.columns.values, precision_mean=w.mean(0).values,
                                           precision_sd=w.std(0, ddof=1).values, precision_low=w.quantile(.025).values,
                                           precision_high=w.quantile(.975).values, chance=chance)))
            row = dict(learner=learner, cancer=n, model=mdl, auprc=float(df[df.model == mdl]["auprc"].mean()), chance=chance)
            for g in (0.2, 0.4, 0.6, 0.8):
                row[f"precision_at_recall_{g}"] = float(w[w.columns[np.argmin(np.abs(w.columns.values - g))]].mean())
            summ.append(row)
    pd.concat(curve, ignore_index=True).to_csv(f"{out}/tables/PR_Curve_{learner}.csv", index=False)
    summ = pd.DataFrame(summ); summ.to_csv(f"{out}/tables/PR_Summary_{learner}.csv", index=False)
    return summ

def save_pr_figure(cfg, cancers, ev_all, out, learner):
    """Fig_PRCurve_<learner>: precision-recall curves (mean over seeds, shaded = 95% range over seeds) for baseline, prior only and full. Primary analysis."""
    names = list(cancers); cols = [PAL["blue"], PAL["orange"]]
    armcol = dict(baseline=PAL["gray"], prior_only=PAL["green"], full=PAL["blue"])
    labels = {"baseline": "Baseline", "prior_only": "Prior only", "full": "Baseline + prior (full)"}
    clean = lambda ax: [ax.spines[s].set_visible(False) for s in ("top", "right")]
    fig, axes = plt.subplots(1, len(names), figsize=(4.4 * len(names) + .4, 4.2), dpi=200, sharey=True, squeeze=False)
    for ax, n, col in zip(axes[0], names, cols):
        crv = ev_all[("PR", n, cfg["primary_ratio"])]; df = ev_all[(n, cfg["primary_ratio"])]
        chance = float((df.test_pos / (df.test_pos + df.test_unk)).mean())
        for mdl in [m for m in ABL_ORDER if m in set(crv.model)]:
            w = crv[crv.model == mdl].pivot(index="seed", columns="recall", values="precision"); c_ = armcol[mdl] or col
            ax.plot(w.columns, w.mean(0), color=c_, lw=1.9, label=labels[mdl])
            ax.fill_between(w.columns, w.quantile(.025), w.quantile(.975), color=c_, alpha=.15, lw=0)
        ax.axhline(chance, color="#666", lw=.9, ls="--", label=f"Chance ({chance:.2f})")
        ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.set_xlabel("Recall"); ax.set_title(n, fontsize=10); clean(ax)
    axes[0][0].set_ylabel("Precision"); axes[0][0].legend(frameon=False, fontsize=7, loc="upper right")
    fig.suptitle(f"{LEARNER_NAMES[learner]}: precision-recall curves (mean over seeds, shaded = 95% range)", fontsize=9)
    plt.tight_layout(); _save(f"{out}/figures/Fig_PRCurve_{learner}.png")

def save_ablation_figures(cfg, cancers, ev_all, out, learner):
    """Fig_AUPRC (bar graph: AUPRC of baseline and prior, both cancers) and Fig_Ablation (baseline, prior only, full). Primary analysis only."""
    names = list(cancers); cols = [PAL["blue"], PAL["orange"]]
    primary = {n: ev_all[(n, cfg["primary_ratio"])] for n in names}
    chance = 1 / (1 + cfg["primary_ratio"])                                  # AUPRC expected from random scores
    clean = lambda ax: [ax.spines[s].set_visible(False) for s in ("top", "right")]
    # --- Fig_AUPRC: bar graph of AUPRC (y axis) for Baseline and Prior (x axis), one bar per cancer; error bars = 95% bootstrap CI
    fig, ax = plt.subplots(figsize=(5.4, 3.9), dpi=200); w_ = .36
    for j, (n, col) in enumerate(zip(names, cols)):
        w = primary[n].pivot(index="seed", columns="model", values="auprc")
        for i, mdl in enumerate(("baseline", "full")):                         # x positions: 0 = Baseline, 1 = Prior
            m, lo, hi = boot_ci(w[mdl].values, cfg["boot"]); x = i + (j - .5) * w_
            ax.bar(x, m, width=w_ * .92, color=col, label=n if i == 0 else None)
            ax.errorbar(x, m, yerr=[[m - lo], [hi - m]], color="black", capsize=4, lw=1.3)
            ax.text(x, hi + .015, f"{m:.2f}", ha="center", va="bottom", fontsize=8)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Baseline", "Prior"]); ax.set_xlim(-.6, 1.6); ax.set_ylim(0, 1.0)
    ax.set_ylabel("AUPRC"); ax.legend(frameon=False, fontsize=8, loc="upper left"); clean(ax)
    ax.set_title(LEARNER_NAMES[learner], fontsize=9); plt.tight_layout(); _save(f"{out}/figures/Fig_AUPRC_{learner}.png")
    # --- Fig_Ablation: baseline / prior only / full model
    labels = {"baseline": "Baseline", "prior_only": "Prior\nonly", "full": "Baseline +\nprior (full)"}
    arms = [m for m in ABL_ORDER if all(m in primary[n].model.values for n in names)]
    armcol = dict(baseline=PAL["gray"], prior_only=PAL["green"], full=None)
    fig, axes = plt.subplots(1, len(names), figsize=(3.6 * len(names) + .6, 3.9), dpi=200, sharey=True, squeeze=False)
    for ax, n, col in zip(axes[0], names, cols):
        w = primary[n].pivot(index="seed", columns="model", values="auprc")
        for i, mdl in enumerate(arms):
            v = w[mdl].values; m, lo, hi = boot_ci(v, cfg["boot"]); c_ = armcol[mdl] or col
            ax.bar(i, m, width=.62, color=c_, alpha=.85, edgecolor="white", lw=.5)
            ax.scatter(i + np.random.default_rng(i).uniform(-.18, .18, len(v)), v, s=7, color="black", alpha=.3, zorder=3)
            ax.errorbar(i, m, yerr=[[m - lo], [hi - m]], color="black", capsize=4, lw=1.4, zorder=4)
        ax.axhline(chance, color="#666", lw=.8, ls="--"); ax.set_ylim(0, 1.02)
        ax.set_xticks(range(len(arms))); ax.set_xticklabels([labels[m] for m in arms], fontsize=7.5); ax.set_title(n, fontsize=10); clean(ax)
    axes[0][0].set_ylabel("AUPRC (mean, 95% CI; dots = seeds)")
    fig.suptitle(LEARNER_NAMES[learner], fontsize=9); plt.tight_layout(); _save(f"{out}/figures/Fig_Ablation_{learner}.png")

def save_delta_bar_figure(cfg, cancers, ev_all, out, learner):
    """Fig_DeltaAUPRC_Bar: bar graph of delta AUPRC = AUPRC(prior) - AUPRC(baseline) for each cancer (mean over seeds, 95% bootstrap CI)."""
    names = list(cancers); cols = [PAL["blue"], PAL["orange"]]
    fig, ax = plt.subplots(figsize=(5.4, 4.6), dpi=200)
    stats_ = []
    for i, n in enumerate(names):
        df = ev_all[(n, cfg["primary_ratio"])]; w = df.pivot(index="seed", columns="model", values="auprc")
        d = (w["full"] - w["baseline"]).values                                 # delta AUPRC per seed = AUPRC(prior) - AUPRC(baseline)
        m, lo, hi = boot_ci(d, cfg["boot"]); stats_.append((m, lo, hi))
        ax.bar(i, m, width=.55, color=cols[i % len(cols)]); ax.errorbar(i, m, yerr=[[m - lo], [hi - m]], color="black", capsize=5, lw=1.3)
    top = max([s[2] for s in stats_] + [0]); bottom = min([s[1] for s in stats_] + [0])
    pad = max(top - bottom, 1e-3) * .18                                         # room above/below for the value labels, so nothing is cut off
    ax.set_ylim(bottom - pad * (1.6 if bottom < 0 else .3), top + pad * 1.6)    # the y axis always shows every bar, every error bar and every label
    for i, (m, lo, hi) in enumerate(stats_):                                    # value label just beyond the end of the error bar (above if positive, below if negative)
        if m >= 0: ax.text(i, hi + pad * .25, f"{m:+.5f}", ha="center", va="bottom", fontsize=9)
        else:      ax.text(i, lo - pad * .25, f"{m:+.5f}", ha="center", va="top", fontsize=9)
    ax.axhline(0, color="#444", lw=.8); ax.set_xticks(range(len(names))); ax.set_xticklabels(names)
    ax.set_ylabel("Delta AUPRC\n(prior - baseline)"); ax.set_xlim(-.6, len(names) - .4)
    ax.yaxis.set_major_formatter(plt.FormatStrFormatter("%.3f"))
    [ax.spines[s].set_visible(False) for s in ("top", "right")]
    ax.set_title(LEARNER_NAMES[learner], fontsize=9); plt.tight_layout(); _save(f"{out}/figures/Fig_DeltaAUPRC_Bar_{learner}.png")

def save_sparse_rich_figure(cfg, cancers, audit, ev_all, out, learner):
    """Fig_SparseRich_<learner>: sparse vs rich comparison in three panels (primary analysis, one bar per cancer):
    (a) known miRNA-disease pairs, (b) AUPRC of baseline and prior (mean over seeds, 95% bootstrap CI), (c) delta AUPRC = AUPRC(prior) - AUPRC(baseline) with its 95% CI."""
    names = list(cancers); cols = [PAL["blue"], PAL["orange"], PAL["green"], PAL["gray"]]
    pairs = [audit.get(f"pairs_{cancers[n]['disease']}") for n in names]
    few = min(p for p in pairs if p is not None)                                   # the cancer with the fewest known pairs is the "sparse" one
    ticks = [f"{n}\n({'sparse' if p == few else 'rich'})" for n, p in zip(names, pairs)]
    fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.6), dpi=200)
    clean = lambda ax: [ax.spines[s].set_visible(False) for s in ("top", "right")]
    # (a) number of known pairs
    ax = axes[0]; ax.bar(range(len(names)), pairs, color=cols[:len(names)], width=.55)
    for i, p in enumerate(pairs): ax.text(i, p + max(pairs) * .02, str(p), ha="center", va="bottom", fontsize=9)
    ax.set_ylim(0, max(pairs) * 1.15); ax.set_ylabel("Known miRNA-disease pairs (HMDD)"); ax.set_title("(a) Known associations", fontsize=10)
    # (b) AUPRC of baseline and prior, and (c) delta AUPRC
    w_ = .36; stats_ = []; hi_max = 0
    for j, n in enumerate(names):
        w = ev_all[(n, cfg["primary_ratio"])].pivot(index="seed", columns="model", values="auprc")
        for i, mdl in enumerate(("baseline", "full")):
            m, lo, hi = boot_ci(w[mdl].values, cfg["boot"]); x = i + (j - (len(names) - 1) / 2) * w_; hi_max = max(hi_max, hi)
            axes[1].bar(x, m, width=w_ * .92, color=cols[j], label=n if i == 0 else None, alpha=1 if i == 1 else .55)
            axes[1].errorbar(x, m, yerr=[[m - lo], [hi - m]], color="black", capsize=4, lw=1.3)
            axes[1].text(x, hi + .012, f"{m:.5f}", ha="center", va="bottom", fontsize=7.5)
        stats_.append(boot_ci((w["full"] - w["baseline"]).values, cfg["boot"]))
    ax = axes[1]; ax.set_xticks([0, 1]); ax.set_xticklabels(["Baseline", "Prior"]); ax.set_xlim(-.6, 1.6); ax.set_ylim(0, min(1.05, hi_max * 1.15 + .03))
    ax.yaxis.set_major_formatter(plt.FormatStrFormatter("%.5f"))                 # y-axis tick labels with 5 digits after the decimal point
    ax.set_ylabel("AUPRC (mean, 95% CI)"); ax.set_title("(b) AUPRC: baseline vs prior", fontsize=10); ax.legend(frameon=False, fontsize=8, loc="upper left")
    ax = axes[2]
    for i, (m, lo, hi) in enumerate(stats_):
        ax.bar(i, m, width=.55, color=cols[i]); ax.errorbar(i, m, yerr=[[m - lo], [hi - m]], color="black", capsize=5, lw=1.3)
    top = max([s[2] for s in stats_] + [0]); bottom = min([s[1] for s in stats_] + [0]); pad = max(top - bottom, 1e-3) * .18
    ax.set_ylim(bottom - pad * (1.6 if bottom < 0 else .3), top + pad * 1.6)
    for i, (m, lo, hi) in enumerate(stats_):
        if m >= 0: ax.text(i, hi + pad * .25, f"{m:+.5f}", ha="center", va="bottom", fontsize=9)
        else:      ax.text(i, lo - pad * .25, f"{m:+.5f}", ha="center", va="top", fontsize=9)
    ax.axhline(0, color="#444", lw=.8); ax.set_ylabel("Delta AUPRC\n(prior - baseline)"); ax.set_title("(c) Gain from the prior", fontsize=10)
    ax.yaxis.set_major_formatter(plt.FormatStrFormatter("%.5f"))
    for ax_ in (axes[0], axes[2]): ax_.set_xticks(range(len(names))); ax_.set_xticklabels(ticks, fontsize=8.5); ax_.set_xlim(-.6, len(names) - .4)
    for ax_ in axes: clean(ax_)
    fig.suptitle(f"Sparse vs rich comparison, {LEARNER_NAMES[learner]}", fontsize=11); plt.tight_layout(); _save(f"{out}/figures/Fig_SparseRich_{learner}.png")

def qc_figures(name, c, raw, X, y, out):
    """Sample/gene quality-control figures for ONE cancer: missingness, PCA, variance, correlation heatmap."""
    g = c["gse"]; clean = lambda ax: [ax.spines[s].set_visible(False) for s in ("top", "right")]
    # (a) missing values per sample, before imputation
    miss = raw.apply(pd.to_numeric, errors="coerce").isna().mean(axis=0) * 100
    fig, ax = plt.subplots(figsize=(4.4, 3.2), dpi=200); ax.hist(miss, bins=20, color=PAL["blue"])
    ax.set_xlabel("Missing values per sample (%)"); ax.set_ylabel("Samples"); clean(ax)
    plt.tight_layout(); _save(f"{out}/figures/Fig_Missingness_{g}.png")
    # (b) PCA of samples, colored by tumor/control, with variance explained on the axes
    pca = PCA(n_components=2); co = pca.fit_transform(X.values.T)
    fig, ax = plt.subplots(figsize=(5, 3.8), dpi=200)
    for val, lab, col in ((1, "Tumor", PAL["orange"]), (0, "Control", PAL["blue"])):
        ax.scatter(co[(y.values == val), 0], co[(y.values == val), 1], s=14, alpha=.8, color=col, label=lab)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% of variance)"); ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% of variance)")
    ax.legend(frameon=False); clean(ax); plt.tight_layout(); _save(f"{out}/figures/Fig_PCA_{g}.png")
    # (c) gene variance distribution
    fig, ax = plt.subplots(figsize=(4.8, 3.2), dpi=200); ax.hist(X.var(axis=1), bins=50, color=PAL["blue"])
    ax.set_xlabel("Gene variance across samples"); ax.set_ylabel("Genes"); clean(ax)
    plt.tight_layout(); _save(f"{out}/figures/Fig_GeneVariance_{g}.png")
    # (d) correlation heatmap of the 100 most variable genes
    fig, ax = plt.subplots(figsize=(5.2, 4.4), dpi=200)
    im = ax.imshow(np.corrcoef(X.values[:100]), cmap="coolwarm", vmin=-1, vmax=1); fig.colorbar(im, ax=ax, label="Correlation")
    ax.set_xlabel("Top 100 variable genes"); ax.set_ylabel("Top 100 variable genes"); plt.tight_layout()
    _save(f"{out}/figures/Fig_CorrelationHeatmap_{g}.png")
def network_figures(name, c, X, net, out):
    """Network figures for ONE cancer: adjacency, dissimilarity, dendrogram with module colors, module-trait heat strip."""
    from scipy.cluster.hierarchy import dendrogram
    import matplotlib.colors
    g = c["gse"]; clean = lambda ax: [ax.spines[s].set_visible(False) for s in ("top", "right")]
    # (e) adjacency and dissimilarity (1 - TOM) for the first 50 genes
    for key, cmap, title, tag in (("adj_head", "viridis", "Adjacency (first 50 genes)", "Adjacency"),
                                  ("diss_head", "magma", "Dissimilarity 1 - TOM (first 50 genes)", "Dissimilarity")):
        fig, ax = plt.subplots(figsize=(4.6, 4), dpi=200); im = ax.imshow(net[key], cmap=cmap); fig.colorbar(im, ax=ax); ax.set_title(title, fontsize=9)
        plt.tight_layout(); _save(f"{out}/figures/Fig_{tag}_{g}.png")
    # (f) gene dendrogram: thin tree lines, zoomed to where the merges happen, with a module color bar and legend (grey = unassigned)
    from matplotlib.collections import LineCollection
    from matplotlib.patches import Patch
    import sys; _lim = sys.getrecursionlimit(); sys.setrecursionlimit(max(_lim, 50000))   # scipy walks the tree recursively; 5,000 genes is deeper than the default limit (1,000)
    try: dd = dendrogram(net["linkage"], no_plot=True)                                       # only computes leaf order and line coordinates
    finally: sys.setrecursionlimit(_lim)
    labs = net["labels"][dd["leaves"]]; n = len(labs); W = 10 * n                             # each leaf occupies 10 x-units
    order = pd.Series(net["labels"][net["labels"] > 0]).value_counts().index.tolist()        # modules, largest first
    pal = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#bcbd22", "#17becf", "#003f5c",
           "#ffa600", "#7a5195", "#ef5675", "#00a676", "#6d4c41", "#4e79a7", "#f28e2b", "#59a14f", "#b07aa1", "#76b7b2"]   # 20 distinct colors, no greys
    mcol = {m: pal[i % len(pal)] for i, m in enumerate(order)}
    rgb = np.array([(0.82, 0.82, 0.82) if l == 0 else matplotlib.colors.to_rgb(mcol[l]) for l in labs])[None, :, :]
    hts = np.concatenate([np.asarray(d, float) for d in dd["dcoord"]]); lo = hts[hts > 0].min(); hi = hts.max()
    fig = plt.figure(figsize=(11, 5.8), dpi=200)
    ax = fig.add_axes([0.08, 0.28, 0.90, 0.64]); bar = fig.add_axes([0.08, 0.17, 0.90, 0.09])
    ax.add_collection(LineCollection([list(zip(x, y)) for x, y in zip(dd["icoord"], dd["dcoord"])], colors="#333333", linewidths=0.4))
    ax.set_xlim(0, W); ax.set_ylim(max(lo - 0.02, 0), hi + 0.01); ax.set_xticks([])
    ax.set_ylabel("Dissimilarity (1 - TOM)", fontsize=9); ax.set_title(f"{name} ({g}): gene dendrogram, {n:,} genes, {len(order)} modules", fontsize=10); clean(ax)
    bar.imshow(rgb, aspect="auto", extent=(0, W, 0, 1), interpolation="nearest"); bar.set_xlim(0, W); bar.set_yticks([]); bar.set_xticks([])
    bar.set_ylabel("Module", rotation=0, ha="right", va="center", fontsize=8)
    for s_ in bar.spines.values(): s_.set_visible(False)
    handles = [Patch(color=mcol[m], label=f"Module {m}") for m in order[:12]] + [Patch(color=(0.82, 0.82, 0.82), label="Unassigned")]
    fig.legend(handles=handles, loc="lower center", ncol=7, fontsize=7, frameon=False, title=("Largest 12 modules shown" if len(order) > 12 else None), title_fontsize=7)
    _save(f"{out}/figures/Fig_Dendrogram_{g}.png")
    # (g) module-trait heat strip for ALL modules (* = passed the FDR and |r| thresholds)
    mt = net["module_trait"].sort_values("module"); selected = set(net["selected"].module) if not net["selected_is_fallback"] else set()
    fig, ax = plt.subplots(figsize=(max(5, 0.45 * len(mt)), 2.2), dpi=200)
    im = ax.imshow(mt.r.values[None, :], cmap="coolwarm", vmin=-1, vmax=1, aspect="auto"); fig.colorbar(im, ax=ax, label="r")
    ax.set_xticks(range(len(mt))); ax.set_xticklabels([f"{m}{'*' if m in selected else ''}" for m in mt.module], fontsize=6, rotation=90)
    ax.set_yticks([0]); ax.set_yticklabels(["Tumor vs control"], fontsize=7)
    if len(mt) <= 30:
        for j, v in enumerate(mt.r.values): ax.text(j, 0, f"{v:.2f}", ha="center", va="center", fontsize=6)
    ax.set_xlabel("Module"); plt.tight_layout(); _save(f"{out}/figures/Fig_ModuleTraitHeatmap_{g}.png")
def enrichment_figure(name, c, enr, out):
    """Network picture of the 15 best miRNA-module enrichments for ONE cancer."""
    g = c["gse"]
    # (h) miRNA-module enrichment network: the 15 best enrichments (solid = q < 0.05, dashed = not significant)
    if len(enr):
        top = enr.head(15); fig, ax = plt.subplots(figsize=(6.4, 4.6), dpi=200)
        mods = sorted(top.module.unique()); mirs = list(dict.fromkeys(top.mirna))
        pos = {("M", m): (0, -i) for i, m in enumerate(mods)}
        pos.update({("R", r): (1, -i * len(mods) / max(len(mirs), 1)) for i, r in enumerate(mirs)})
        for _, r in top.iterrows():
            (x0, y0), (x1, y1) = pos[("M", r.module)], pos[("R", r.mirna)]
            ax.plot([x0, x1], [y0, y1], color=PAL["green"] if r.q < 0.05 else "#999999", ls="-" if r.q < 0.05 else "--",
                    lw=0.8 + min(r.overlap, 12) / 4, alpha=.7)
        for (kind, k), (x, yy) in pos.items():
            ax.scatter([x], [yy], s=260 if kind == "M" else 60, color=PAL["blue"] if kind == "M" else PAL["orange"], zorder=3)
            ax.text(x - .03 if kind == "M" else x + .03, yy, f"Module {k}" if kind == "M" else k,
                    ha="right" if kind == "M" else "left", va="center", fontsize=7)
        ax.set_xlim(-.45, 1.5); ax.axis("off"); n_sig = int((enr.q < 0.05).sum())
        ax.set_title(f"{name}: top miRNA-module enrichments ({n_sig} with q < 0.05; dashed = not significant)", fontsize=8)
        plt.tight_layout(); _save(f"{out}/figures/Fig_EnrichmentNetwork_{g}.png")

def save_exploratory_figures(name, c, raw, X, y, net, enr, out):
    """All per-cancer exploratory figures in one call (used by run_all; the step-by-step cells call the three parts separately)."""
    qc_figures(name, c, raw, X, y, out); network_figures(name, c, X, net, out); enrichment_figure(name, c, enr, out)

def package_results(cfg, log, out):
    """Write the run log (all settings and counts), a checksum for every file, and the downloadable zip."""
    def clean(o):                                                     # convert numpy types so they can be saved as JSON
        if isinstance(o, dict): return {str(k): clean(v) for k, v in o.items()}
        if isinstance(o, (list, tuple)): return [clean(v) for v in o]
        if isinstance(o, (np.integer,)): return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, (np.bool_,)): return bool(o)
        return o
    json.dump(clean(dict(config=cfg, **log)), open(f"{out}/run_log.json", "w"), indent=2, default=str)
    files = sorted(os.path.join(dp, f) for dp, _, fs in os.walk(out) for f in fs)
    manifest = {f: sha256(f) for f in files}; json.dump(manifest, open(f"{out}/manifest_sha256.json", "w"), indent=2)
    with zipfile.ZipFile("SparseNet_MDA_Cancer_Results.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for f in files + [f"{out}/manifest_sha256.json"]: z.write(f)

def run_all(cfg, cancers, L, expected_pairs=None):
    """Run the complete workflow (Steps 2-9) and return the main results in a dictionary."""
    out = cfg["outdir"]; os.makedirs(f"{out}/figures", exist_ok=True); os.makedirs(f"{out}/tables", exist_ok=True)
    log = {}                                                          # everything printed/saved in run_log.json
    H, audit = audit_associations(L, cancers, expected_pairs, log)    # Step 2: association audit
    targets = L["targets"](); log["n_target_pairs"] = int(len(targets)); print("miRTarBase-restricted target pairs:", len(targets))
    prep, nets, enrs, priors = {}, {}, {}, {}
    for name, c in cancers.items():                                   # Steps 3-6, once per cancer
        prep[name], nets[name], enrs[name], priors[name] = prepare_cancer_arm(name, c, L, targets, H, cfg, out)
    log["preparation"] = prep
    ev_all, tests, delta = {}, {}, {}
    for lrn in cfg["learners"]:                                       # Steps 7-8 once per classifier (random forest, logistic regression)
        ev_all[lrn] = evaluate_all(H, cancers, priors, cfg, out, lrn)                       # Step 7: baseline vs prior evaluation
        tests[lrn], delta[lrn] = save_tables(cfg, cancers, H, audit, ev_all[lrn], enrs, priors, out, lrn)   # Step 8: statistics and tables
        save_figures(cfg, cancers, audit, delta[lrn], nets, out, lrn)                       # figures
        save_ablation_figures(cfg, cancers, ev_all[lrn], out, lrn)                          # Fig_AUPRC and Fig_Ablation
        save_delta_bar_figure(cfg, cancers, ev_all[lrn], out, lrn)                          # Fig_DeltaAUPRC_Bar
        save_sparse_rich_figure(cfg, cancers, audit, ev_all[lrn], out, lrn)                 # Fig_SparseRich
        save_pr_figure(cfg, cancers, ev_all[lrn], out, lrn)                                 # Fig_PRCurve
    package_results(cfg, log, out)                                    # log, checksums, zip
    return dict(audit=audit, prep=prep, tests=tests, delta=delta, enrich=enrs, nets=nets, H=H)

## Cell 10 — Step 2: audit the miRNA–disease associations (HMDD)
**What it does:** upload the HMDD v4.0 file when promted.HMDD file needs to be first downloaded from the website http://www.cuilab.cn/hmdd downloads section and that file is uploaded when promted, counts the unique human miRNA–disease pairs for each cancer, and builds the **association matrix `H`** (rows = miRNA families, columns = diseases, 1 = known association). `H` is the graph the baseline model learns from.
**Workflow step:** Step 2 — data audit. **Manuscript:** Authenticated Data Audit.
**What you will see:** a table with the pair counts and a summary of how many diseases each miRNA is linked to.
**How to read it:** `pairs_after_name_harmonization` can be a little lower than `hmdd_pairs` because different spellings of the same miRNA family (for example `hsa-mir-21` and `hsa-miR-21-5p`) are merged.

In [ ]:
OUT = CFG["outdir"]
os.makedirs(f"{OUT}/figures", exist_ok=True); os.makedirs(f"{OUT}/tables", exist_ok=True)
LOG = {}                                     # collects every count; saved later as run_log.json
H, AUDIT = audit_associations(LOADERS, CANCERS, EXPECTED_HMDD_PAIRS, LOG)
rows = []
for name, c in CANCERS.items():
    d = c["disease"]
    rows.append(dict(disease=d, hmdd_pairs=AUDIT.get(f"pairs_{d}")))
print("\n--- HMDD pairs per disease ---")
display(pd.DataFrame(rows))
print(f"HMDD evidence rows: {AUDIT['entries_total']:,}; "
      f"unique human miRNA-disease pairs: {AUDIT['human_pairs_total']:,}")
print(f"Association matrix H: {H.shape[0]:,} miRNA families x {H.shape[1]:,} diseases")
print("\n--- How many diseases is each miRNA family linked to? ---")
display(H.sum(axis=1).describe().to_frame("diseases per miRNA").T.round(5))

## Cell 11 — Step 1b: download the validated miRNA–target pairs
**What it does:** downloads experimentally supported miRNA→gene pairs from OmniPath, keeping only records that come from miRTarBase. These tell us which genes each miRNA is known to regulate (used for enrichment in Cell 15).
**Workflow step:** Step 1 — data acquisition (second half). **Manuscript:** Materials and Data Sources.
**What you will see:** the first rows of the table and three counts (pairs, distinct miRNAs, distinct genes).
**How to read it:** a very small number of pairs or miRNAs means few miRNAs can ever receive a prior score.

In [ ]:
TARGETS = LOADERS["targets"]()
LOG["n_target_pairs"] = int(len(TARGETS))
print("miRTarBase-restricted target pairs:", len(TARGETS))
display(TARGETS.head())
tf = TARGETS["mirna"].map(mirna_family)
print(f"{len(TARGETS):,} miRNA-gene pairs | {tf.nunique():,} distinct miRNA families | {TARGETS['gene'].nunique():,} distinct target genes")

## Cell 12 — Step 3a: download expression data and label tumor vs control
**What it does:** for each cancer, downloads the GEO expression matrix (a table of probes × samples) and reads each sample's description to decide whether it is a **tumor (1)** or a **control (0)**. If the counts do not match the study design (104/59 and 104/17), the run **stops with an error** instead of guessing.
**Workflow step:** Step 3 (start) — data preparation. **Manuscript:** Materials and Data Sources (Expression Cohorts).
**What you will see:** a table of the sample counts found versus the design, then a few example samples with their assigned label for each cancer.


In [ ]:
RAW, META, Y = {}, {}, {}
rows = []
for name, c in CANCERS.items():
    raw, meta = LOADERS["expr"](c["gse"]); y = label_samples(meta, c["gse"])
    counts = dict(tumor=int((y == 1).sum()), control=int((y == 0).sum()), excluded=int(y.isna().sum()))
    print(f"{name} ({c['gse']}) sample labels:", counts)
    ok = (counts["tumor"], counts["control"]) == (c["expect"]["tumor"], c["expect"]["control"])
    rows.append(dict(cancer=name, geo_series=c["gse"], probes=raw.shape[0], samples_in_file=raw.shape[1], tumor=counts["tumor"],
                     control=counts["control"], excluded=counts["excluded"], design_tumor=c["expect"]["tumor"],
                     design_control=c["expect"]["control"], design_match="yes" if ok else "NO"))
    if not ok:                                # stop if the labels do not match the study design
        raise RuntimeError(f"Sample labels for {c['gse']} do not match the design {c['expect']}: inspect `meta` and adjust label_samples().")
    META[name] = meta.assign(label=y)
    y = y.dropna(); RAW[name] = raw[y.index]; Y[name] = y     # keep only labelled samples
print("\n--- Samples found vs study design ---")
display(pd.DataFrame(rows))
for name in CANCERS:
    print(f"\n--- {name}: example samples and their assigned label (1 = tumor, 0 = control, NaN = excluded) ---")
    display(META[name].groupby("label", dropna=False).head(3).sort_values("label"))

## Cell 13 — Step 3b: prepare the expression data and remove outlier samples
**What it does:** turns the raw probe values into a clean **gene × sample** table: log-transform if needed, quantile-normalize, map probes to gene symbols, keep the most informative genes (at most 5,000), then remove samples that are outliers by **both** network connectivity and a robust PCA test.
**Workflow step:** Step 3 — data preparation. **Manuscript:** Dataset-Preparation Procedure.
**What you will see:** a table with the counts at each stage (probes mapped → genes → genes kept → samples kept) and which samples were removed, followed by quality-control plots for each cancer: missing values per sample, PCA of samples (orange = tumor, blue = control), gene variance, and a correlation heatmap of the 100 most variable genes.
**How to read it:** in the PCA plot, a visible separation of orange and blue along PC1 or PC2 means tumor and control differ strongly in expression (good for the module–trait step). Samples far from all others are potential outliers. `samples_removed` should be small; a large number would mean the QC rule needs review.

In [ ]:
XS, PREP = {}, {}
for name, c in CANCERS.items():
    X, info = prepare_expression(RAW[name], LOADERS["probe2gene"](c["gpl"]), CFG)
    keep, qc = sample_qc(X); X = X[keep]; y = Y[name][keep]      # drop outlier samples
    info.update(qc); info.update(samples_retained=int(X.shape[1]), tumor=int((y == 1).sum()), control=int((y == 0).sum()))
    XS[name], Y[name], PREP[name] = X, y, info
print("--- Preparation summary ---")
display(pd.DataFrame({n: {"probes mapped to a gene": p["probes_mapped"], "genes after merging probes": p["genes_after_collapse"],
                          "genes after filtering": p["genes_after_filter"], "genes used in network": p["genes_used"],
                          "log2 applied": p["log2_applied"], "missing values (fraction)": round(p["missing_fraction"], 5),
                          "samples retained": p["samples_retained"], "tumor": p["tumor"], "control": p["control"],
                          "samples removed": ", ".join(p["removed"]) if p["removed"] else "none"} for n, p in PREP.items()}))
for name, c in CANCERS.items():
    print(f"\n=== Quality-control plots: {name} ===")
    qc_figures(name, c, RAW[name], XS[name], Y[name], OUT)

## Cell 14 — Steps 4–5: co-expression network, gene modules, and the link to tumor status
**What it does:** for each cancer it
1. computes how strongly every pair of genes moves together, and picks the **soft-threshold power β** that makes the network resemble a real biological network (a few highly connected genes, many weakly connected ones);
2. builds the network, clusters genes by shared neighborhoods, and cuts the tree into **modules** (groups of co-expressed genes);
3. summarizes each module by its **eigengene** (its main expression pattern) and correlates it with tumor status; p-values are corrected for the number of modules tested (`fdr`).
**Workflow steps:** 4 (network and modules) and 5 (module–trait association). **Manuscript:** Proposed Method (soft adjacency, topological overlap, modules, eigengene–trait association).
**What you will see:** per cancer: the table of powers tried with their scale-free fit (the chosen β is printed with a warning if the fit criterion was **not** met and a fallback was used); the module–tumor correlation table; and plots: adjacency and dissimilarity matrices (first 50 genes), the gene dendrogram with a module color bar, and a heat strip of every module's correlation with tumor status (`*` = passed the thresholds).
**How to read it:** `r` is the module–tumor correlation (−1 to 1; the sign is arbitrary because the eigengene's direction is arbitrary) and `fdr` is the adjusted p-value. A module is "tumor-associated" if `fdr < 0.05` and `|r| ≥ 0.20`. **Very large modules** (thousands of genes) are less informative for the later enrichment step. A final summary table compares the two cancers.

In [ ]:
NETS = {}
for name, c in CANCERS.items():
    net = build_network(XS[name], Y[name], CFG); NETS[name] = net
    net["module_trait"].to_csv(f"{OUT}/tables/ModuleTrait_{c['gse']}.csv", index=False)
    net["power_table"].to_csv(f"{OUT}/tables/SoftThreshold_{c['gse']}.csv", index=False)
    print(f"\n=================== {name} ===================")
    print(f"Chosen soft-threshold power beta = {net['beta']}" +
          ("  (WARNING: scale-free criterion R^2 >= 0.80 NOT met; the best available power was used = fallback)" if net["beta_fallback"] else "  (criterion met)"))
    print("\n--- Powers tried (signed_r2 = scale-free fit; mean_k = mean connectivity) ---")
    display(net["power_table"].round(5))
    print("\n--- Modules and their correlation with tumor status (sorted by strength) ---")
    mt = net["module_trait"].assign(abs_r=lambda d: d.r.abs()).sort_values("abs_r", ascending=False).drop(columns="abs_r")
    display(mt.head(15).round(5))
    network_figures(name, c, XS[name], net, OUT)
print("\n--- Network summary ---")
display(pd.DataFrame({n: {"soft-threshold power": v["beta"], "fallback used": v["beta_fallback"], "initial modules": v["n_initial_modules"],
                          "modules after merging": v["n_modules"], "genes not in any module": v["n_grey"],
                          "largest module (genes)": int(v["module_trait"]["size"].max()),
                          "tumor-associated modules": int(0 if v["selected_is_fallback"] else len(v["selected"]))} for n, v in NETS.items()}))

## Cell 15 — Step 6: miRNA–module enrichment and the prior
**What it does:** for every tumor-associated module and every miRNA, tests whether the miRNA's validated target genes are **over-represented** in that module (hypergeometric test, then false-discovery correction → `q`). Each miRNA's best score becomes its **prior** (0 if it was never enriched).
**Workflow step:** Step 6. **Manuscript:** Candidate Master-Regulator Enrichment.
**What you will see:** per cancer: the 10 strongest enrichments, how many tests were run, **how many have q < 0.05**, how many miRNAs received a non-zero prior, and a network picture of the top 15 enrichments (solid green line = significant, dashed grey = not significant).
**How to read it:** `overlap` = how many of the miRNA's targets fall inside the module; `q` is the corrected p-value. **If no enrichment reaches q < 0.05, the prior is built from non-significant signals**, so it should not be described as evidence that specific miRNAs regulate the module.

In [ ]:
ENR, PRIORS = {}, {}
for name, c in CANCERS.items():
    net = NETS[name]
    enr = module_enrichment(net, TARGETS, XS[name].index, CFG)
    prior = make_prior(enr, H.index)
    ENR[name], PRIORS[name] = enr, prior
    PREP[name].update(beta=net["beta"], beta_fallback=net["beta_fallback"], n_modules=net["n_modules"],
                      n_initial_modules=net["n_initial_modules"], n_grey_genes=net["n_grey"],
                      n_significant_modules=int(0 if net["selected_is_fallback"] else len(net["selected"])),
                      selected_is_fallback=net["selected_is_fallback"],
                      n_enrichments_q05=int((enr["q"] < 0.05).sum()) if len(enr) else 0,
                      n_mirnas_with_prior=int((prior > 0).sum()))
    print(f"\n=================== {name} ===================")
    print(f"Enrichment tests run: {len(enr):,} | with q < 0.05: {PREP[name]['n_enrichments_q05']} | miRNAs with a non-zero prior: {PREP[name]['n_mirnas_with_prior']}")
    if len(enr):
        print("\n--- Ten strongest miRNA-module enrichments ---")
        display(enr.head(10).round(5))
    else:
        print("No miRNA had any target inside the selected modules.")
    enrichment_figure(name, c, enr, OUT)
LOG["preparation"] = PREP

## Cell 16 — Step 7: the evaluation — does the prior improve prediction?
**What it does:** the core experiment, run once for **each classifier** (random forest and logistic regression). For each cancer and each of 30 random seeds it splits the known miRNAs into training / test, trains the classifier, and measures **AUPRC** (area under the precision–recall curve) on the held-out test pairs for:
- `baseline` — the classifier trained on each miRNA's associations with *other* diseases only;
- `full` — the same plus the co-expression prior as an extra feature, i.e. the **AUPRC(prior)** model;
- `prior_only` — the prior alone (ablation).
The cell also repeats this at the 1:1 and 10:1 unknown-to-positive ratios. **Workflow step:** 7. **Manuscript:** Experimental Design, Leakage-Prevention Strategy.
**What you will see:** the average split sizes for each classifier and model, then the first rows of the per-seed results (one AUPRC per seed and model). The ΔAUPRC itself is computed in the next cell.


In [ ]:
EV = {}
for LRN in CFG["learners"]:                                  # 30 seeds x 3 ratios x 2 cancers x 3 models, for each classifier
    print(f"Evaluating with {LEARNER_NAMES[LRN]} ...")
    EV[LRN] = evaluate_all(H, CANCERS, PRIORS, CFG, OUT, LRN)
primary_all = pd.concat([EV[l][(n, CFG["primary_ratio"])] for l in CFG["learners"] for n in CANCERS], ignore_index=True)
print(f"--- Average split sizes per classifier and model (primary analysis, {CFG['primary_ratio']} unknowns per positive) ---")
display(primary_all.groupby(["learner", "cancer", "model"])[["train_pos", "train_unk", "test_pos", "test_unk"]].mean().round(5))
print("\n--- First rows of the per-seed test results ---")
display(primary_all.head(8).round(5))

## Cell 17 — Step 8: ΔAUPRC for each cancer
**What it does:** computes the one result for each cancer and for each classifier (random forest and logistic regression):
**ΔAUPRC = AUPRC(prior) − AUPRC(baseline)**, calculated for each of the 30 seeds, then averaged, with a **95% bootstrap confidence interval** and a one-sided paired Wilcoxon test of whether ΔAUPRC > 0. It also writes all result tables (CSV, GraphML) to the output folder.
**Workflow step:** Step 8. **Manuscript:** Evaluation Metrics and Statistical Analysis; Results.
**What you will see:** for each classifier, one table with a row per cancer (ΔAUPRC, its CI and p-value, the mean AUPRC without and with the prior, and in how many of the 30 seeds the prior helped), then the same table at the 1:1 and 10:1 ratios, then the **precision-recall summary** (AUPRC and precision at recall 0.2 / 0.4 / 0.6 / 0.8 for baseline, prior only and full, next to the chance level; `PR_Summary_<learner>.csv`; the full curve is in `PR_Curve_<learner>.csv`).
**How to read it:** ΔAUPRC > 0 with a CI above zero means the prior improved held-out prediction in that cancer. Compare the two rows: the hypothesis predicts a larger ΔAUPRC in cholangiocarcinoma (sparse) than in breast (rich). Because the 30 seeds re-split the same data, the intervals and p-values are optimistic and should be described as such.
**Sparse vs rich table:** `Sparse_Rich_Comparison_<learner>.csv` lists, for each cancer, the known HMDD pairs, the mean AUPRC of the baseline, the mean AUPRC with the prior, and ΔAUPRC (prior − baseline).

In [ ]:
TESTS, DELTA = {}, {}
for LRN in CFG["learners"]:
    TESTS[LRN], DELTA[LRN] = save_tables(CFG, CANCERS, H, AUDIT, EV[LRN], ENR, PRIORS, OUT, LRN)
    print(f"\n--- ΔAUPRC = AUPRC(prior) - AUPRC(baseline), primary analysis, {LEARNER_NAMES[LRN]} ---")
    display(TESTS[LRN].set_index("cancer").round(5))
    print(f"\n--- Same quantity at the other unknown-to-positive ratios, {LEARNER_NAMES[LRN]} ---")
    display(pd.read_csv(f"{OUT}/tables/Sensitivity_Ratios_{LRN}.csv").round(5))
    print(f"\n--- Sparse vs rich comparison (known pairs, AUPRC baseline, AUPRC prior, delta AUPRC), {LEARNER_NAMES[LRN]} ---")
    display(pd.read_csv(f"{OUT}/tables/Sparse_Rich_Comparison_{LRN}.csv").round(5))
    print(f"\n--- Precision-recall summary (primary analysis), {LEARNER_NAMES[LRN]} ---")
    display(pd.read_csv(f"{OUT}/tables/PR_Summary_{LRN}.csv").round(5))

## Cell 18 — Result figures
**What it does:** Result figures (the classifier-specific ones once for each classifier, with the classifier named in the title and file name) and shows them here: number of known pairs (sparse vs rich); ΔAUPRC for each of the 30 seeds with its 95% interval; and, for each cancer, the strongest module–tumor correlations and the module size distribution. It also draws **Fig_PRCurve** (precision-recall curves of baseline, prior only and full for both cancers; mean over seeds with a shaded 95% range, dashed line = chance), **Fig_DeltaAUPRC_Bar** (bar graph of ΔAUPRC for each cancer with its 95% interval), **Fig_AUPRC** (bar graph: AUPRC of baseline and prior for both cancers) **Fig_Ablation** (similarity only, prior only, full model) and **Fig_SparseRich** (sparse vs rich comparison in three panels: known pairs, AUPRC of baseline vs prior, and ΔAUPRC with its 95% interval).
**Workflow step:** Results figures. **Manuscript:** Results.
**How to read it:** in the ΔAUPRC plot, points above the horizontal zero line are seeds where the prior helped; the black bar is the mean with its 95% interval.

In [ ]:
for LRN in CFG["learners"]:
    save_figures(CFG, CANCERS, AUDIT, DELTA[LRN], NETS, OUT, LRN)
    save_ablation_figures(CFG, CANCERS, EV[LRN], OUT, LRN)      # Fig_AUPRC_<learner>.png and Fig_Ablation_<learner>.png
    save_delta_bar_figure(CFG, CANCERS, EV[LRN], OUT, LRN)      # Fig_DeltaAUPRC_Bar_<learner>.png
    save_sparse_rich_figure(CFG, CANCERS, AUDIT, EV[LRN], OUT, LRN)   # Fig_SparseRich_<learner>.png
    save_pr_figure(CFG, CANCERS, EV[LRN], OUT, LRN)             # Fig_PRCurve_<learner>.png

## Cell 19 — Random forest vs logistic regression: accuracy, precision at recall, F1, ROC AUC
**What it does:** re-scores both classifiers on the same 30 seeds and the same train/test splits as Cell 16 and computes, for each cancer: accuracy (plus balanced accuracy), precision, recall and F1 at a fixed 0.5 threshold; precision at recall 0.2 / 0.4 / 0.6 / 0.8; ROC AUC; and AUPRC. It shows **one summary table** (mean ± SD over seeds) and draws **one combined ROC comparison** and **one combined precision-recall comparison** for the two classifiers.
**Settings:** `ARM = "full"` compares the model with the prior; set `ARM = "baseline"` to compare without it. `THRESH = 0.5` is the fixed decision threshold (not tuned on test data).
**How to read it:** with 5 unknowns per positive, always predicting "not associated" gives about 0.83 accuracy, so read accuracy together with balanced accuracy, F1 and the precision-recall numbers. Run it after Cell 18 and before Cell 20, so its tables and figures are included in the single results zip made in Cell 20.
**Files written:** `tables/Model_Comparison_Table.csv`, `Model_Comparison_Table_mean_sd.csv`, `Model_Comparison_PerSeed.csv`, `figures/Fig_ROC_Comparison.png`, `Fig_PRCurve_Comparison.png`.

In [ ]:
# =====================================================================
# Needs PRIORS, H, CANCERS, CFG, OUT (all created before Cell 18)
# Cell 19 - Random forest vs Logistic regression: accuracy, precision at recall, F1, ROC AUC,
#           one summary table, one combined ROC figure, one combined PR-curve figure.
# =====================================================================
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
                             recall_score, f1_score, roc_curve)

ARM = "full"        # which feature set to compare: "full" (profile + prior), "baseline" (profile only) or "prior_only"
THRESH = 0.5        # fixed decision threshold for accuracy / precision / recall / F1 (not tuned on test data)
RATIO = CFG["primary_ratio"]
FPR_GRID = np.linspace(0, 1, 101)
RECALL_POINTS = (0.2, 0.4, 0.6, 0.8)
LCOL = {"rf": PAL["blue"], "logreg": PAL["orange"]}

def split_scores(H, disease, feats, seed, ratio, cfg, learner, arm):
    """Same split logic as one_split (same seeds, same test pairs), but returns the test labels and scores of ONE arm."""
    rng = np.random.default_rng(seed)
    idx_all = np.arange(len(H)); y = H[disease].values == 1
    pos = rng.permutation(idx_all[y]); unk = rng.permutation(idx_all[~y])
    n_te = max(1, round(.2 * len(pos))); te_p, tr_p = pos[:n_te], pos[n_te:]
    n_te_u = min(ratio * len(te_p), len(unk) // 2)
    te_u = unk[:n_te_u]; tr_u = unk[n_te_u:n_te_u + ratio * len(tr_p)]
    tr_idx = np.r_[tr_p, tr_u]; tr_y = np.r_[np.ones(len(tr_p)), np.zeros(len(tr_u))]
    te_idx = np.r_[te_p, te_u]; te_y = np.r_[np.ones(len(te_p)), np.zeros(len(te_u))]
    assert not (set(tr_idx) & set(te_idx))
    F = feats[arm]
    mdl = make_model(learner, cfg, seed).fit(F[tr_idx], tr_y)
    return te_y, mdl.predict_proba(F[te_idx])[:, 1]

rows, roc_curves, pr_curves = [], {}, {}
for name, c in CANCERS.items():
    dis = [col for col in H.columns if col.lower() == c["disease"].lower()][0]
    feats = feature_sets(H, dis, PRIORS[name])
    for lrn in CFG["learners"]:
        tprs, precs = [], []
        for s in range(CFG["n_seeds"]):
            te_y, sc = split_scores(H, dis, feats, CFG["master_seed"] + s, RATIO, CFG, lrn, ARM)
            pred = (sc >= THRESH).astype(int)
            fpr, tpr, _ = roc_curve(te_y, sc)
            tprs.append(np.interp(FPR_GRID, fpr, tpr)); tprs[-1][0] = 0.0
            pr = pr_on_grid(te_y, sc); precs.append(pr)
            row = dict(cancer=name, learner=LEARNER_NAMES[lrn], seed=s,
                       accuracy=accuracy_score(te_y, pred),
                       balanced_accuracy=balanced_accuracy_score(te_y, pred),
                       precision=precision_score(te_y, pred, zero_division=0),
                       recall=recall_score(te_y, pred, zero_division=0),
                       f1=f1_score(te_y, pred, zero_division=0),
                       roc_auc=roc_auc_score(te_y, sc),
                       auprc=average_precision_score(te_y, sc),
                       chance_precision=te_y.mean())
            for g in RECALL_POINTS:
                row[f"precision_at_recall_{g}"] = pr[int(round(g * 100))]
            rows.append(row)
        roc_curves[(name, lrn)] = np.array(tprs); pr_curves[(name, lrn)] = np.array(precs)

per_seed = pd.DataFrame(rows)
per_seed.to_csv(f"{OUT}/tables/Model_Comparison_PerSeed.csv", index=False)

# ---------- ONE summary table: mean (SD over 30 seeds) for every metric, both models, both cancers ----------
metric_cols = ["accuracy", "balanced_accuracy", "precision", "recall", "f1", "roc_auc", "auprc"] + \
              [f"precision_at_recall_{g}" for g in RECALL_POINTS] + ["chance_precision"]
agg = per_seed.groupby(["cancer", "learner"])[metric_cols].agg(["mean", "std"])
summary_num = agg.xs("mean", axis=1, level=1).round(5)
summary_num.to_csv(f"{OUT}/tables/Model_Comparison_Table.csv")
summary_txt = pd.DataFrame({m: [f"{agg.loc[i, (m, 'mean')]:.5f} ± {agg.loc[i, (m, 'std')]:.5f}" for i in agg.index]
                            for m in metric_cols}, index=agg.index)
summary_txt.to_csv(f"{OUT}/tables/Model_Comparison_Table_mean_sd.csv")
print(f"--- Random forest vs logistic regression ('{ARM}' feature set, {RATIO}:1 unknown-to-positive, "
      f"threshold {THRESH} for accuracy/precision/recall/F1; mean ± SD over {CFG['n_seeds']} seeds) ---")
display(summary_txt)

# ---------- Figure A: combined ROC comparison ----------
fig, axes = plt.subplots(1, len(CANCERS), figsize=(4.6 * len(CANCERS), 4.4), dpi=200, sharey=True, squeeze=False)
for ax, name in zip(axes[0], CANCERS):
    for lrn in CFG["learners"]:
        T = roc_curves[(name, lrn)]; m, sd = T.mean(0), T.std(0, ddof=1)
        auc_m = per_seed[(per_seed.cancer == name) & (per_seed.learner == LEARNER_NAMES[lrn])]["roc_auc"]
        ax.plot(FPR_GRID, m, color=LCOL[lrn], lw=2, label=f"{LEARNER_NAMES[lrn]} (AUC {auc_m.mean():.3f} ± {auc_m.std(ddof=1):.3f})")
        ax.fill_between(FPR_GRID, np.clip(m - sd, 0, 1), np.clip(m + sd, 0, 1), color=LCOL[lrn], alpha=.15, lw=0)
    ax.plot([0, 1], [0, 1], "--", color="#666", lw=.9, label="Chance")
    ax.set_xlabel("False positive rate"); ax.set_title(name, fontsize=10)
    ax.legend(frameon=False, fontsize=7, loc="lower right"); [ax.spines[s].set_visible(False) for s in ("top", "right")]
axes[0][0].set_ylabel("True positive rate")
fig.suptitle(f"ROC comparison: random forest vs logistic regression ({ARM} model; mean over seeds, shaded = ±1 SD)", fontsize=9)
plt.tight_layout(); _save(f"{OUT}/figures/Fig_ROC_Comparison.png")

# ---------- Figure B: PR-curve comparison ----------
fig, axes = plt.subplots(1, len(CANCERS), figsize=(4.6 * len(CANCERS), 4.4), dpi=200, sharey=True, squeeze=False)
for ax, name in zip(axes[0], CANCERS):
    for lrn in CFG["learners"]:
        P = pr_curves[(name, lrn)]; m, sd = P.mean(0), P.std(0, ddof=1)
        ap = per_seed[(per_seed.cancer == name) & (per_seed.learner == LEARNER_NAMES[lrn])]["auprc"]
        ax.plot(PR_GRID, m, color=LCOL[lrn], lw=2, label=f"{LEARNER_NAMES[lrn]} (AUPRC {ap.mean():.3f} ± {ap.std(ddof=1):.3f})")
        ax.fill_between(PR_GRID, np.clip(m - sd, 0, 1), np.clip(m + sd, 0, 1), color=LCOL[lrn], alpha=.15, lw=0)
    ch = per_seed[per_seed.cancer == name]["chance_precision"].mean()
    ax.axhline(ch, ls="--", color="#666", lw=.9, label=f"Chance ({ch:.2f})")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.set_xlabel("Recall"); ax.set_title(name, fontsize=10)
    ax.legend(frameon=False, fontsize=7, loc="upper right"); [ax.spines[s].set_visible(False) for s in ("top", "right")]
axes[0][0].set_ylabel("Precision")
fig.suptitle(f"Precision-recall comparison: random forest vs logistic regression ({ARM} model; mean over seeds, shaded = ±1 SD)", fontsize=9)
plt.tight_layout(); _save(f"{OUT}/figures/Fig_PRCurve_Comparison.png")

print("Cell 19 done: Model_Comparison_Table*.csv, Model_Comparison_PerSeed.csv, Fig_ROC_Comparison.png, Fig_PRCurve_Comparison.png saved.")

## Cell 20 — Save the results and zip
**What it does:** writes the run log (all settings and counts), a SHA-256 checksum for every file and `SparseNet_MDA_Cancer_Results.zip`.
**Workflow step:** Step 9 — reproducibility. **Manuscript:** Ethical and Reproducibility Considerations; Data and Code Availability.
**What you will see:** a table of every file in the zip with its size.

In [ ]:
package_results(CFG, LOG, OUT)
with zipfile.ZipFile("SparseNet_MDA_Cancer_Results.zip") as z:
    display(pd.DataFrame([(i.filename, f"{i.file_size / 1024:.1f}") for i in z.infolist()], columns=["file in zip", "size (KB)"]))

## Cell 21 — Reading of the results
**What it does:** reads the ΔAUPRC tables (one per classifier)
**How to read it:** a result counts as "improved" only if the 95% interval is entirely above zero **and** p < 0.05.

In [ ]:
c1, c2 = list(CANCERS)
for LRN in CFG["learners"]:
    t = TESTS[LRN].set_index("cancer")
    print(f"=============== {LEARNER_NAMES[LRN]} ===============")
    for c in (c1, c2):
        r = t.loc[c]
        verdict = "IMPROVED" if (r["ci_low"] > 0 and r["p_one_sided"] < 0.05) else "no clear improvement"
        print(f"{c}: ΔAUPRC = {r['delta_auprc']:+.5f} (95% CI {r['ci_low']:+.5f} to {r['ci_high']:+.5f}, one-sided p = {r['p_one_sided']:.5f}); "
              f"AUPRC {r['auprc_baseline']:.5f} -> {r['auprc_prior']:.5f}; prior helped in {int(r['seeds_improved'])} of {int(r['n_seeds'])} seeds -> {verdict}.")
    bigger = c1 if t.loc[c1, "delta_auprc"] > t.loc[c2, "delta_auprc"] else c2
    print(f"The larger gain was in {bigger}. The hypothesis predicts the larger gain in {c1} (the association-sparse cancer): "
          + ("consistent with it." if bigger == c1 else "the opposite of what it predicts."))
    ab = pd.read_csv(f"{OUT}/tables/Ablation_AUPRC_{LRN}.csv")
    print("Ablation (primary analysis; mean AUPRC):")
    for c in CANCERS:
        a = ab[ab.cancer == c].set_index("model")
        print(f" - {c}: baseline {a.loc['baseline','auprc']:.5f} | prior only {a.loc['prior_only','auprc']:.5f} | full {a.loc['full','auprc']:.5f}")
    print()
print("Known limits of this run:")
for c in CANCERS:
    p = PREP[c]
    notes = []
    if p["n_enrichments_q05"] == 0: notes.append("no miRNA-module enrichment reached q < 0.05 (the prior is built from non-significant enrichments)")
    if p["beta_fallback"]: notes.append(f"scale-free criterion not met (fallback power {p['beta']})")
    if p["removed"]: notes.append(f"{len(p['removed'])} outlier sample(s) removed")
    print(f" - {c}: " + ("; ".join(notes) if notes else "no flags"))
print(" - The 30 seeds re-split the same data, so confidence intervals and p-values are optimistic.")
print(" - Unknown miRNA-disease pairs are used as stand-in negatives; they are not confirmed negatives.")

## Cell 22 — Download the results
**What it does:** Download the results to zip file, `SparseNet_MDA_Cancer_Results.zip`, to computer ( Downloads folder). If the download is blocked, open the folder icon on the left of Colab and download the zip from there.


In [ ]:
try:
    from google.colab import files
    files.download("SparseNet_MDA_Cancer_Results.zip")
except ImportError:
    print("Results are in SparseNet_MDA_Cancer_Results.zip")